### Why this exists

Everything in `litesearch.tree` assumes a document whose structure is in its headings and whose
atom is a paragraph. A Sanskrit text is neither, and the mismatch is not cosmetic:

| assumption | what a Sanskrit e-text actually is |
|---|---|
| structure is in the headings | there are usually **no headings** — the structure is a citation stamped on every verse (`ViP_1,1.1`) |
| a chunk boundary can fall anywhere | the verse is **metrically closed**; half a śloka is a fragment, not a proposition |
| whitespace separates words | *sandhi* fuses them (`tat ca` → `tac ca`), so the token a reader types often is not in the text |
| one script, one spelling | the same verse circulates in Devanagari, IAST, SLP1, Harvard-Kyoto and `Krishna`-style English |
| a passage belongs to its document | a śloka travels: Manusmṛti verses sit inside the Mahābhārata, and commentators quote across works |

So this module supplies the four things that follow from those rows — a **transliteration fold**, a
**verse-atomic chunker**, a **reference-driven section tree**, and a **graph whose edges are the
relations Sanskrit texts actually have** (next verse, parallel passage, commentary-on, metre).

It supplies them *through the existing seams*, not beside them. `db.add_sanskrit` is `db.add_doc`
with `tree_fn=verse_tree` and `chunk_fn=sanskrit_chunks`; the fold rides in the store's existing
`metadata` column, which `get_store` already indexes for FTS; the graph writes the tables
`get_graph` already defines. `toc`, `read`, `breadcrumb`, `doc_search`, `sections`, `graph_search`,
`clusters` and `peers` all then work on a Sanskrit corpus with no code of their own.

**What informed the formats.** [GRETIL](https://gretil.sub.uni-goettingen.de/gretil.html) for the
plain-text conventions — the `// Mn_1.1 //` reference, the `[[iti ... adhyāyaḥ]]` colophon, `[h: :h]`
headings, `§X uvāca:` speakers, `[Page I,12]` markers, and commentary files that introduce each
commentator with a bare `śrīdharaḥ :`. The
[Digital Corpus of Sanskrit](http://www.sanskrit-linguistics.org/dcs/) for the CoNLL-U shape and for
the observation that drives `lemma_fn`: with validated lemmas, retrieval stops fighting sandhi.
[Dharmamitra](https://dharmamitra.github.io/dharmamitra-guides/)'s MITRA parallel-passage mining for
the idea that finding *the same passage elsewhere* is a first-class retrieval result — done here
lexically and offline, which within a single language is the cheaper tool, since Sanskrit parallels
are usually verbatim once the fold has removed the orthographic noise.

**No model is required anywhere.** Metre scansion, verse segmentation, the reference tree, parallel
detection and the fold are all deterministic string work, so this runs offline and in CI. Real
lemmatisation and cross-lingual matching are the two places tokens genuinely buy something, and both
are keyword arguments (`lemma_fn=`, `emb_fn=`) with working non-LLM defaults.

In [ ]:
#| default_exp sanskrit

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore.all import Path, patch, first, L, AttrDict, ifnone, merge
from fastlite import Database
from collections import Counter
import json, re, unicodedata

from litesearch.core import _in, _slug, process_content
from litesearch.tree import TreeNode, MIN_CHUNK, build_tree, heading_path, _clean_title
from litesearch.tree import summarize_extractive
from litesearch.data import chunk_markdown, clean
from litesearch.graph import _pmi_edges

## One text, five spellings

The first problem is not retrieval, it is identity. `कृष्ण`, `kṛṣṇa`, `kfzRa` (SLP1), `kRSNa`
(Harvard-Kyoto) and `Krishna` are one word, and FTS5 — `porter unicode61` or the UAX#29 chain
`litesearch` installs — treats them as five.

`fold` is the answer, and its whole design is that it is **one-way and cheap**: strip to an ASCII
skeleton and use that as an extra index key. `sanskrit_chunks` puts the fold of each chunk in the
store's `metadata` column, which `get_store` already registers for FTS alongside `content`, so a
reader typing Devanagari matches the text while a reader typing `dharmaksetre` matches the fold —
with no new column, no second table and no second index.

`loose` is the query-side extension for popular romanisations. It is deliberately aggressive because
`sanskrit_query` ORs it *alongside* the plain fold: a wrong guess costs one dead term in the FTS
expression and can never lose a match.

`detect_scheme` will not guess SLP1 or Harvard-Kyoto unless asked. Both are plain ASCII — `what a
quick fix` is valid SLP1 — so sniffing them silently rewrites an English query into nonsense.

In [ ]:
#| export
# --- Devanagari -> IAST ------------------------------------------------------------------------
_DV_V = {'अ':'a','आ':'ā','इ':'i','ई':'ī','उ':'u','ऊ':'ū','ऋ':'ṛ','ॠ':'ṝ','ऌ':'ḷ','ॡ':'ḹ',
         'ए':'e','ऐ':'ai','ओ':'o','औ':'au','ऎ':'e','ऒ':'o','ऍ':'e','ऑ':'o'}
_DV_M = {'ा':'ā','ि':'i','ी':'ī','ु':'u','ू':'ū','ृ':'ṛ','ॄ':'ṝ','ॢ':'ḷ','ॣ':'ḹ',
         'े':'e','ै':'ai','ो':'o','ौ':'au','ॆ':'e','ॊ':'o','ॅ':'e','ॉ':'o'}
_DV_C = {'क':'k','ख':'kh','ग':'g','घ':'gh','ङ':'ṅ','च':'c','छ':'ch','ज':'j','झ':'jh','ञ':'ñ',
         'ट':'ṭ','ठ':'ṭh','ड':'ḍ','ढ':'ḍh','ण':'ṇ','त':'t','थ':'th','द':'d','ध':'dh','न':'n',
         'प':'p','फ':'ph','ब':'b','भ':'bh','म':'m','य':'y','र':'r','ल':'l','ळ':'ḻ','व':'v',
         'श':'ś','ष':'ṣ','स':'s','ह':'h','ऴ':'ḻ','ऱ':'r','ऩ':'n','क़':'k','ख़':'kh','ग़':'g',
         'ज़':'j','ड़':'ḍ','ढ़':'ḍh','फ़':'ph','य़':'y'}
_DV_S = {'ं':'ṃ','ः':'ḥ','ँ':'m̐','ऽ':"'",'ॐ':'oṃ'}
_DV_P = {'।':'|','॥':'||','॰':'.','‌':'','‍':''}
_VIRAMA, _NUKTA = '्', '़'
DEV_DIGITS = str.maketrans('०१२३४५६७८९', '0123456789')

_DEV_RE  = re.compile(r'[ऀ-ॿ]')
_IAST_RE = re.compile(r'[āīūṛṝḷḹṃṁḥṅñṇṭḍśṣḻ]')
_SLP_RE  = re.compile(r'[fFxXwWqQ]')
_HK_RE   = re.compile(r'lRR|lR|RR|[AIUMHTDNSZGJ]')

def _dev_iast(s):
    'Devanagari to IAST, honouring the implicit `a`, virāma and mātrās. Unknown characters pass through.'
    out, i, n = [], 0, len(s)
    while i < n:
        ch = s[i]
        if ch in _DV_C:
            out.append(_DV_C[ch]); i += 1
            if i < n and s[i] == _NUKTA: i += 1
            if   i < n and s[i] == _VIRAMA: i += 1               # bare consonant, no vowel
            elif i < n and s[i] in _DV_M: out.append(_DV_M[s[i]]); i += 1
            else: out.append('a')                                # the implicit vowel
            continue
        if   ch in _DV_V: out.append(_DV_V[ch])
        elif ch in _DV_M: out.append(_DV_M[ch])                  # stray mātrā
        elif ch in _DV_S: out.append(_DV_S[ch])
        elif ch in _DV_P: out.append(_DV_P[ch])
        elif ch in (_VIRAMA, _NUKTA): pass
        else: out.append(ch.translate(DEV_DIGITS))
        i += 1
    return ''.join(out)

# --- SLP1 / Harvard-Kyoto ---------------------------------------------------------------------
_SLP2I = {'a':'a','A':'ā','i':'i','I':'ī','u':'u','U':'ū','f':'ṛ','F':'ṝ','x':'ḷ','X':'ḹ',
          'e':'e','E':'ai','o':'o','O':'au','M':'ṃ','H':'ḥ','~':'m̐',
          'k':'k','K':'kh','g':'g','G':'gh','N':'ṅ','c':'c','C':'ch','j':'j','J':'jh','Y':'ñ',
          'w':'ṭ','W':'ṭh','q':'ḍ','Q':'ḍh','R':'ṇ','t':'t','T':'th','d':'d','D':'dh','n':'n',
          'p':'p','P':'ph','b':'b','B':'bh','m':'m','y':'y','r':'r','l':'l','v':'v',
          'S':'ś','z':'ṣ','s':'s','h':'h','L':'ḻ'}
_HK2I = {'a':'a','A':'ā','i':'i','I':'ī','u':'u','U':'ū','R':'ṛ','RR':'ṝ','lR':'ḷ','lRR':'ḹ',
         'e':'e','ai':'ai','o':'o','au':'au','M':'ṃ','H':'ḥ',
         'k':'k','kh':'kh','g':'g','gh':'gh','G':'ṅ','c':'c','ch':'ch','j':'j','jh':'jh','J':'ñ',
         'T':'ṭ','Th':'ṭh','D':'ḍ','Dh':'ḍh','N':'ṇ','t':'t','th':'th','d':'d','dh':'dh','n':'n',
         'p':'p','ph':'ph','b':'b','bh':'bh','m':'m','y':'y','r':'r','l':'l','v':'v',
         'z':'ś','S':'ṣ','s':'s','h':'h','L':'ḻ'}
_I2SLP = {}
for _k, _v in _SLP2I.items(): _I2SLP.setdefault(_v, _k)

def _xlit(s, table, maxlen):
    'Longest-match transliteration of `s` through `table`. Unknown characters pass through.'
    out, i, n = [], 0, len(s or '')
    while i < n:
        for k in range(min(maxlen, n-i), 0, -1):
            if (v := table.get(s[i:i+k])) is not None: out.append(v); i += k; break
        else: out.append(s[i]); i += 1
    return ''.join(out)

def detect_scheme(text, romanized:bool=False) -> str:
    """The transliteration `text` is written in: `devanagari`, `iast` or `ascii`.

    With `romanized=True` it will also guess `slp1` vs `hk`, which is a **guess and not a
    detection**: both are plain ASCII, `what a quick fix` is valid SLP1, and mistaking a user's
    English query for SLP1 silently rewrites it into nonsense. Leave it off for query text."""
    t = text or ''
    if _DEV_RE.search(t): return 'devanagari'
    if _IAST_RE.search(t): return 'iast'
    if romanized:
        if _SLP_RE.search(t): return 'slp1'
        if _HK_RE.search(t): return 'hk'
    return 'ascii'

def to_iast(text, scheme:str=None) -> str:
    """IAST for a Sanskrit string.

    `scheme=None` converts Devanagari and leaves romanised input alone — see `detect_scheme` for
    why SLP1 and Harvard-Kyoto have to be named (`scheme='slp1'`) rather than sniffed."""
    sc = scheme or ('devanagari' if _DEV_RE.search(text or '') else 'iast')
    if sc == 'devanagari': return _dev_iast(text or '')
    if sc == 'slp1': return _xlit(text, _SLP2I, 1)
    if sc == 'hk':   return _xlit(text, _HK2I, 3)
    return text or ''

def to_slp1(text, scheme:str=None) -> str:
    'SLP1 for a Sanskrit string: one ASCII character per phoneme, and reversible.'
    return _xlit(to_iast(text, scheme), _I2SLP, 2)

# --- the search key ---------------------------------------------------------------------------
_FOLD_TT = str.maketrans({'ā':'a','ī':'i','ū':'u','ṛ':'r','ṝ':'r','ḷ':'l','ḹ':'l','ṃ':'m','ṁ':'m',
                          'ḥ':'h','ṅ':'n','ñ':'n','ṇ':'n','ṭ':'t','ḍ':'d','ś':'s','ṣ':'s','ḻ':'l',
                          "'":'','’':'','ʼ':'','`':'','\u02bc':''})
_WS = re.compile(r'\s+')

def fold(text, scheme:str=None) -> str:
    """The diacritic-free ASCII skeleton of a Sanskrit string — `kṛṣṇa` and `कृष्ण` both give `krsna`.

    This is the key that makes full-text search scheme-agnostic, and it is why
    `litesearch.sanskrit` needs no schema change: the fold of every chunk goes into the store's
    `metadata` column, which `get_store` already indexes for FTS alongside `content`. A reader who
    types Devanagari matches the content; one who types `dharmaksetre` matches the fold."""
    t = unicodedata.normalize('NFC', to_iast(text, scheme)).translate(_FOLD_TT)
    t = ''.join(c for c in unicodedata.normalize('NFD', t) if not unicodedata.combining(c))
    return t.lower()

# Popular English spellings of Sanskrit, applied to the *query only*. `Krishna` and `Vishnu` are
# how readers type; `krsna` and `visnu` are what `fold` stored. A wrong conflation here costs a
# little precision on one query and can never corrupt the index.
_LOOSE = (('ksh','ks'), ('sh','s'), ('chh','ch'), ('aa','a'), ('ee','i'), ('ii','i'),
          ('oo','u'), ('uu','u'), ('w','v'), ('z','j'), ('rr','r'))
# `Krishna`, `Trishna`, `Sanskrit`: a consonant-flanked `ri` is almost always ṛ.
_LOOSE_RI = re.compile(r'(?<=[bcdfghjklmnpqstvxz])ri(?=[bcdfghjklmnpqstvxz])')

def loose(text, scheme:str=None) -> str:
    """Query-side fold that also absorbs popular romanisations: `Krishna` -> `krsna`, `Geeta` -> `gita`.

    Aggressive on purpose. `sanskrit_query` ORs this variant *alongside* the plain fold rather than
    in place of it, so a wrong guess (`priya` -> `prya`) costs one dead term in the FTS query and
    can never lose a match the fold would have found."""
    t = fold(text, scheme)
    for a, b in _LOOSE: t = t.replace(a, b)
    return _LOOSE_RI.sub('r', t)

In [ ]:
# one word, five spellings, one key
assert fold('कृष्ण') == fold('kṛṣṇa') == fold(to_iast('kfzRa','slp1')) == 'krsna'
assert fold(to_iast('kRSNa', 'hk')) == 'krsna'
assert loose('Krishna') == 'krsna' and loose('Bhagavad Geeta') == fold('bhagavad gītā')
assert to_slp1('kṛṣṇa') == 'kfzRa' and to_iast(to_slp1('rāmāyaṇam ṛṣiḥ jñānam'), 'slp1') == 'rāmāyaṇam ṛṣiḥ jñānam'
assert to_iast('धर्मक्षेत्रे कुरुक्षेत्रे') == 'dharmakṣetre kurukṣetre'
# the implicit `a`, the virama and the matra
assert to_iast('राम') == 'rāma' and to_iast('रम्') == 'ram' and to_iast('रामः') == 'rāmaḥ'
# an English query is never mistaken for SLP1 unless you insist
assert detect_scheme('what a quick fix') == 'ascii'
assert detect_scheme('what a quick fix', romanized=True) == 'slp1'   # exactly why it is opt-in
assert detect_scheme('कृष्ण') == 'devanagari' and detect_scheme('kṛṣṇa') == 'iast'

## Metre, before segmentation

Scansion comes first because segmentation depends on it. It looks like a philologist's ornament and
is load-bearing for three reasons:

1. it is the **verse/prose classifier** this module needs anyway — a unit whose syllables do not
   divide into four comparable pādas is bhāṣya, and wants a different chunker;
2. it is a **facet** worth having (`verse_search(..., meter='śārdūlavikrīḍita')`);
3. it is a **graph node** that links verses no shared vocabulary would.

The rules are the classical ones: a syllable is *guru* when its vowel is long, when anusvāra or
visarga follows, or when two or more consonants follow before the next vowel. The trap is that the
aspirates are single consonants — `atha` is light-light, `artha` is heavy-light — which is why the
consonant table is matched longest-first.

The patterns are **derived from the gaṇas** rather than typed out. A hand-copied 21-syllable
sragdharā pattern is a silent bug nobody ever finds.

In [ ]:
#| export
_VOW = ('ai','au','ā','ī','ū','ṝ','ḹ','e','o','a','i','u','ṛ','ḷ')   # longest first: `ai` before `a`
_LONG = {'ā','ī','ū','ṝ','ḹ','e','o','ai','au'}                       # `e` and `o` are long in Sanskrit
_CONS2 = ('kh','gh','ch','jh','ṭh','ḍh','th','dh','ph','bh')          # one consonant each, not two
_CONS1 = set('kgṅcjñṭḍṇtdnpbmyrlvśṣshḻ')
_CODA  = set('ṃḥ')

def syllables(text, scheme:str=None) -> L:
    """The syllables of a Sanskrit string as `(syllable, guru)` pairs.

    A syllable is heavy (*guru*) when its vowel is long, when it carries anusvāra or visarga, or
    when **two or more** consonants follow before the next vowel. The aspirates are the trap: `kh`
    and `dh` are single consonants, so `atha` is light-light and `artha` is heavy-light. That is
    why the consonant table is matched longest-first rather than character by character."""
    s = ''.join(c for c in to_iast(text, scheme) if c.isalpha() or c in _CODA)
    units, i, n = [], 0, len(s)
    while i < n:
        if (v := first(_VOW, lambda x: s.startswith(x, i))): units.append(('V', v)); i += len(v)
        elif s[i:i+2] in _CONS2:                             units.append(('C', s[i:i+2])); i += 2
        elif s[i] in _CONS1 or s[i] in _CODA:                units.append(('C', s[i])); i += 1
        else: i += 1
    out, onset = L(), ''
    for j, (k, u) in enumerate(units):
        if k == 'C': onset += u; continue
        run = []
        for k2, u2 in units[j+1:]:
            if k2 == 'V': break
            run.append(u2)
        heavy = (u in _LONG) or any(x in _CODA for x in run) or len(run) >= 2
        out.append((onset + u, heavy)); onset = ''
    return out

# The eight gaṇas, plus the two single-syllable fillers. Patterns are *derived* from these rather
# than typed out, because a hand-copied 21-syllable pattern is a silent bug nobody ever finds.
_GANA = dict(m='ggg', y='lgg', r='glg', s='llg', t='ggl', j='lgl', bh='gll', n='lll', g='g', l='l')
def _pat(spec): return ''.join(_GANA[x] for x in spec.split())

# The common classical metres, by gaṇa recipe. `anuṣṭubh` is handled separately: it is a syllable
# *count* with a cadence rule, not a fixed pattern.
METERS = {
    'śālinī':           _pat('m t t g g'),
    'indravajrā':       _pat('t t j g g'),
    'upendravajrā':     _pat('j t j g g'),
    'rathoddhatā':      _pat('r n r l g'),
    'svāgatā':          _pat('r n bh g g'),
    'vaṃśastha':        _pat('j t j r'),
    'indravaṃśā':       _pat('t t j r'),
    'drutavilambita':   _pat('n bh bh r'),
    'toṭaka':           _pat('s s s s'),
    'bhujaṅgaprayāta':  _pat('y y y y'),
    'praharṣiṇī':       _pat('m n j r g'),
    'rucirā':           _pat('j bh s j g'),
    'vasantatilakā':    _pat('t bh j j g g'),
    'mālinī':           _pat('n n m y y'),
    'pṛthvī':           _pat('j s j s y l g'),
    'mandākrāntā':      _pat('m bh n t t g g'),
    'śikhariṇī':        _pat('y m n s bh l g'),
    'hariṇī':           _pat('n s m r s l g'),
    'śārdūlavikrīḍita': _pat('m s j s t t g'),
    'sragdharā':        _pat('m r bh n y y y'),
}
# upajāti is not a metre but a licence: any mix of these two across the four pādas.
_MIXED = {'upajāti': ('indravajrā', 'upendravajrā')}

def _pat_ok(w, pat):
    'Does a pāda of weights match a gaṇa pattern? The final syllable is anceps, as always.'
    if len(w) != len(pat): return False
    return all(x == (c == 'g') for x, c in zip(w[:-1], pat[:-1]))

def detect_meter(text, scheme:str=None, weights:list=None):
    """The metre of a verse: `AttrDict(name, syllables, per_pada, pathya)`. `name` is None when the
    shape is recognisable but the pattern is not in `METERS`, and the whole call is None for
    anything too short to be a verse at all.

    Worth having for three reasons beyond display. It is a **facet** ("show me the
    śārdūlavikrīḍita verses"), it is a **graph node** that links verses no shared vocabulary would,
    and it is the **verse/prose classifier** this module needs anyway: a unit whose syllables do
    not divide into four comparable pādas is bhāṣya, not a śloka, and wants a different chunker."""
    w = weights if weights is not None else [g for _, g in syllables(text, scheme)]
    n = len(w)
    if n < 8: return None
    if n % 4 == 0:
        q = n // 4
        quarters = [w[i*q:(i+1)*q] for i in range(4)]
        if q == 8:
            # anuṣṭubh. pathyā is the ordinary cadence of the even pādas: 5-6-7 = ˘ ¯ ˘
            return AttrDict(name='anuṣṭubh', syllables=n, per_pada=8,
                            pathya=all(x[4:7] == [False, True, False] for x in quarters[1::2]))
        for nm, p in METERS.items():
            if len(p) == q and all(_pat_ok(x, p) for x in quarters):
                return AttrDict(name=nm, syllables=n, per_pada=q, pathya=None)
        for nm, (a, b) in _MIXED.items():
            pa, pb = METERS[a], METERS[b]
            if len(pa) == q and all(_pat_ok(x, pa) or _pat_ok(x, pb) for x in quarters):
                return AttrDict(name=nm, syllables=n, per_pada=q, pathya=None)
    return AttrDict(name=None, syllables=n, per_pada=(n//4 if n % 4 == 0 else None), pathya=None)

def is_verse(text, scheme:str=None, weights:list=None, lo:int=8, hi:int=26) -> bool:
    'True when a unit scans as four comparable pādas — the test that separates śloka from bhāṣya.'
    m = detect_meter(text, scheme, weights)
    return bool(m and (m.name or (m.per_pada and lo <= m.per_pada <= hi)))

In [ ]:
# the aspirate rule: `kh`/`th` are one consonant, so `atha` is light-light and `artha` is heavy-light
assert [g for _, g in syllables('atha')]  == [False, False]
assert [g for _, g in syllables('artha')] == [True,  False]
assert [g for _, g in syllables('rāmaḥ')] == [True,  True]   # long vowel, then visarga

# Meghaduta 1.1 is mandakranta (17x4); the Sarasvati stotra is sardulavikridita (19x4).
# `syllables` drops the dandas and everything else that is not a phoneme, so raw text is fine.
MD = '''kaścit kāntāvirahaguruṇā svādhikārātpramattaḥ
śāpenāstaṃgamitamahimā varṣabhogyeṇa bhartuḥ /
yakṣaścakre janakatanayāsnānapuṇyodakeṣu
snigdhacchāyātaruṣu vasatiṃ rāmagiryāśrameṣu //'''
SD = '''yā kundendutuṣārahāradhavalā yā śubhravastrāvṛtā
yā vīṇāvaradaṇḍamaṇḍitakarā yā śvetapadmāsanā /
yā brahmācyutaśaṃkaraprabhṛtibhir devaiḥ sadā vanditā
sā māṃ pātu sarasvatī bhagavatī niḥśeṣajāḍyāpahā //'''
m = detect_meter(MD)
assert (m.name, m.syllables, m.per_pada) == ('mandākrāntā', 68, 17), m
m = detect_meter(SD)
assert (m.name, m.per_pada) == ('śārdūlavikrīḍita', 19), m

# the patterns are derived from the ganas, so their lengths are arithmetic rather than typing
assert len(METERS['sragdharā']) == 21 and len(METERS['vasantatilakā']) == 14
assert len(METERS['indravajrā']) == len(METERS['upendravajrā']) == 11
assert detect_meter('rāmaḥ') is None                      # too short to be a verse at all

## Segmenting verse

The verse is the atom, so segmentation is the whole game — and Sanskrit e-texts are unusually
generous about it. A single daṇḍa (`/`, `।`) closes a hemistich, a double daṇḍa (`//`, `॥`) closes
the verse, and GRETIL puts the citation between two more.

`split_verses` also returns the things that are *not* verses, because they are what
`verse_tree` uses to find sections in the many texts that have no headings: colophons
(`iti ... adhyāyaḥ`), GRETIL's `[h: :h]` headings, and speaker changes (`X uvāca`).

Scansion is what classifies each unit. A run of text between two daṇḍas that does not divide into
four comparable pādas is prose — Vedic prose, sūtra, bhāṣya — and `sanskrit_chunks` sends it to a
different chunker. One detail worth pointing at: `arjuna uvāca` stays in the verse text — it is what a reader searches
for — but it is excluded from the metrical scan, because a stage direction is not a pāda and
counting it makes every line of dialogue in the Gītā read as prose.

In [ ]:
#| export
# Sanskrit division words in `fold`ed form, ranked in the same space as `tree._STRUCT_RANK`: a
# lower number sits higher in the tree. The ranks are a *prior* — `verse_tree` compacts whatever
# the document actually uses into consecutive levels, exactly as `struct_levels` does for prose.
SANSKRIT_UNITS = {
    # whole-work divisions
    'kanda':0, 'khanda':0, 'parva':0, 'amsa':0, 'astaka':0, 'mandala':0, 'skandha':0, 'sthana':0,
    'adhikarana':0, 'vibhaga':0, 'lambaka':0, 'kalpa':0, 'satka':0, 'pancasika':0,
    # chapter-level
    'adhyaya':1, 'sarga':1, 'sukta':1, 'prapathaka':1, 'anuvaka':1, 'pariccheda':1, 'prakarana':1,
    'ullasa':1, 'taranga':1, 'ucchvasa':1, 'valli':1, 'brahmana':1, 'prasna':1, 'ahnika':1,
    'upadesa':1, 'nirnaya':1, 'stabaka':1, 'vimana':1, 'siddhi':1, 'dasaka':1, 'sataka':1,
    'varga':1, 'pada':1, 'parisista':1, 'uddesa':1, 'nikaya':1,
    # section-level
    'kandika':2, 'patala':2, 'paryaya':2, 'vidhi':2, 'ahnaya':2, 'stotra':2, 'anga':2,
    # leaf-level
    'sloka':3, 'sutra':3, 'karika':3, 'mantra':3, 'gatha':3, 'vakya':3, 'rc':3, 'khila':3,
}
# Sanskrit ordinals, folded, as they appear in colophons: `iti ... prathamo 'dhyayah`.
_ORDINALS = {o: i+1 for i, o in enumerate(
    'prathama dvitiya trtiya caturtha pancama sastha saptama astama navama dasama ekadasa dvadasa '
    'trayodasa caturdasa pancadasa sodasa saptadasa astadasa ekonavimsa vimsa ekavimsa dvavimsa '
    'trayovimsa caturvimsa pancavimsa sadvimsa saptavimsa astavimsa ekonatrimsa trimsa'.split())}

# The nominal endings a division word actually turns up in. A whitelist rather than "up to three
# characters of anything", because the loose version lets `pada` claim `padma`.
_ENDINGS = frozenset(('', 'a','ah','am','an','as','at','au','aih','ais','aya','ayah','anam','ani',
                      'asya','ena','e','esu','i','ih','in','ini','is','o','os','u','ya','yam',
                      'ni','su','bhih','bhyah','ayoh','ayam'))

def _unit_rank(w):
    """The `(unit, rank)` of a folded word if it is a division word, else None.

    Three kinds of slack, every one of them load-bearing on real colophons. A case ending is
    matched against the **stem** rather than the dictionary form, because an ending replaces the
    stem-final `a` instead of following it (`amsa` + locative is `amse`, which does not start with
    `amsa`), and what follows the stem has to be in `_ENDINGS`. And an initial `a`
    is restored, because `fold` drops the avagraha and the commonest colophon in the corpus —
    `prathame \'mse prathamo \'dhyayah` — is nothing but two elided initial `a`s.

    The stem is also matched **compound-finally**, since that is how the big divisions are named:
    the Ramayana closes a section with `balakande`, the Mahabharata with `adiparvani`, and neither
    word begins with its own division word. Only stems of four characters or more are allowed to
    match inside a word, which is what stops `pada` claiming `padma`."""
    if not w or len(w) < 3: return None
    best = None
    for cand in (w, 'a' + w):
        for k, r in SANSKRIT_UNITS.items():
            stem = k[:-1] if k.endswith('a') else k
            i = cand.rfind(stem)
            if i < 0 or cand[i+len(stem):] not in _ENDINGS: continue
            if i > 0 and len(stem) < 4: continue        # `pada` may not match inside a compound
            if best is None or len(stem) > len(best[2]): best = (k, r, stem)
    return (best[0], best[1]) if best else None

def unit_words(text, scheme:str=None) -> L:
    """The division words in a line, in the order they appear.

    The colophon of a Purāṇa or an epic names the *whole ladder* it is closing —
    `iti ... prathame 'mse prathamo 'dhyayah` is aṃśa then adhyāya — which is how `verse_tree`
    learns what to call the levels of a text that carries no headings at all."""
    out = L()
    for w in re.findall(r'[a-z]+', fold(text, scheme)):
        if (u := _unit_rank(w)) and (not out or out[-1] != u[0]): out.append(u[0])
    return out

# `iti ... adhyayah` — the colophon that closes a section, and names it. GRETIL wraps it in
# brackets; editions print it bare between daṇḍas.
_COLOPHON = re.compile(r'^[\s\[\]|/।॥*#]*((?:iti|samapta)\b.{0,220}?)[\s\[\]|/।॥*#]*$', re.I)
# `[h: ... :h]` / `[k: ... :k]` — GRETIL's own heading and colophon markers.
_GHEAD  = re.compile(r'\[\s*h\s*:\s*(.*?)\s*:\s*h\s*\]', re.S)
_GKOLO  = re.compile(r'\[\s*k\s*:\s*(.*?)\s*:\s*k\s*\]', re.S)
_PAGEMK = re.compile(r'\[\s*(?:page|p\.|fol\.|folio)[^\]]{0,40}\]', re.I)
# `parasara uvaca:` — a change of speaker. GRETIL marks some with a leading `§`.
_SPEAKER = re.compile(r'^[\s§|/।॥]*([^\s|/।॥].{0,50}?)\s+uv[āa]ca\s*[:\s]*$', re.I)

# The double daṇḍa closes a verse, the single one a hemistich. `_DANDA2` captures its match, so a
# verse can be handed back with the mark it was printed with rather than a normalised one.
_DANDA2 = re.compile(r'(\|\||//|॥|।।)')
_DANDA1 = re.compile(r'\||/|।')  # non-capturing: pādas are split on it, it is not kept
# `Mn_1.1`, `ViP_1,1.1`, `MS_1,1.1`, `BhG 3.1`, `12.34ab`, `1`
_VNUM = re.compile(r'^(?:([A-Za-zÀ-ɏḀ-ỿ][A-Za-z0-9À-ɏḀ-ỿ]{0,14})[_ ] ?)?(\d+(?:[.,]\d+){0,4})([a-z]{0,3})$')
# A body line ends in a daṇḍa, optionally followed by a reference between two more.
_BODY_LINE = re.compile(r'[|/।॥](?:[\w.,ऀ-ॿ-]{0,25}\s*[|/।॥]+)?\s*$')

def parse_ref(s):
    """`ViP_1,1.1` -> `AttrDict(siglum='ViP', num=(1,1,1), seps=(',','.'), pada='')`, else None.

    The separators are kept because they are part of the citation: `ViP_1,1` is book 1 chapter 1
    and `ViP_1.1` is not the same address, so a node title built from a prefix has to round-trip
    them rather than normalise everything to a dot."""
    t = (s or '').strip().translate(DEV_DIGITS)
    if not t or len(t) > 40: return None
    if not (m := _VNUM.match(t)): return None
    return AttrDict(siglum=m.group(1) or '', num=tuple(int(x) for x in re.split(r'[.,]', m.group(2))),
                    seps=tuple(re.findall(r'[.,]', m.group(2))), pada=m.group(3) or '')

def format_ref(siglum, num, seps=()) -> str:
    'Render `("ViP", (1,1))` back as `ViP_1,1` — the citation a Sanskritist writes.'
    if not num: return siglum or ''
    out = str(num[0])
    for i, n in enumerate(num[1:]): out += (seps[i] if i < len(seps) else '.') + str(n)
    return f'{siglum}_{out}' if siglum else out

def unit_heading(ln, scheme:str=None):
    """`AttrDict(unit, n, level)` when a line is a Sanskrit division heading, else None.

    Matches `Sarga 3`, `## adhyāyaḥ 3`, `॥ अध्याय ३ ॥` and the colophon form
    `iti ... tṛtīyo 'dhyāyaḥ`, in any transliteration, because the line is folded before matching.
    A number (Arabic, Devanagari or a Sanskrit ordinal) is **required**: without one, `padārtha`
    reads as the division word `pada` and every philosophical verse becomes a chapter break."""
    raw = (ln or '').strip()
    if not raw or len(raw) > 160: return None
    f = fold(raw, scheme).strip(' |/.:-*#[]()')
    if not f or len(f) > 100: return None
    words = re.findall(r'[a-z]+', f)
    if not words or len(words) > 12: return None
    hit = first((u for w in reversed(words) if (u := _unit_rank(w))), None)
    if not hit: return None
    nums = re.findall(r'\d+', f)
    n = int(nums[-1]) if nums else first(
        (v for w in words for k, v in _ORDINALS.items() if w.startswith(k[:-1])), None)
    if n is None: return None
    return AttrDict(unit=hit[0], n=n, level=hit[1])

# A line of Sanskrit carries letters, daṇḍas, avagraha and hyphens — and no digits, colons,
# parentheses or full stops, which is what every line of a bibliography has.
_SANS_LINE = re.compile(r"^[A-Za-zÀ-ɏḀ-ỿऀ-ॿ'’\u02bc\- \t|/।॥]+$")
# ... and these are the words a GRETIL header is made of.
_HDR_WORDS = frozenset('the of and by for on in to with a an version text plain based edition ed '
                       'see list from this input encoding proofread typed analyzed revised copyright '
                       'university press vol volume ff pp date file header'.split())

def strip_header(text, max_scan:int=300, max_back:int=12) -> str:
    """Drop a GRETIL-style front matter block: everything before the first line of actual text.

    The block has no delimiter to look for — it is free-form bibliography, an encoding table and a
    licence — so the boundary is found from the other side: a line of a Sanskrit text ends in a
    daṇḍa and a line of a bibliography does not.

    Then it **backs up**, and that second half is not optional. Only the *hemistich* carries the
    daṇḍa, so in any four-line metre the opening pāda has no mark on it at all: cutting at the first
    daṇḍa-terminated line silently eats the first quarter of the first verse, which then fails to
    scan and is filed as prose. So the walk continues backwards over lines that look like Sanskrit
    and stops at the first one that looks like a bibliography."""
    lns = (text or '').splitlines()
    for i, ln in enumerate(lns[:max_scan]):
        s = ln.strip()
        if not s or 'http' in s.lower() or '@' in s: continue
        if not _BODY_LINE.search(s): continue
        j, back = i, 0
        while j > 0 and back < max_back:
            k = j - 1
            while k >= 0 and not lns[k].strip(): k -= 1     # blank lines are not a boundary
            if k < 0: break
            prev = lns[k].strip()
            if not _SANS_LINE.match(prev) or len(prev.split()) < 3 or prev.isupper(): break
            if sum(w.lower() in _HDR_WORDS for w in prev.split()) >= 2: break
            j, back = k, back + 1
        return '\n'.join(lns[j:])
    return text or ''

In [ ]:
#| export
def _mk(kind, text, page, ref=None, **kw):
    r = parse_ref(ref) if ref else None
    return AttrDict(kind=kind, text=(text or '').strip(), page=page, pos=0,
                    ref=(ref or '').strip(), siglum=(r.siglum if r else ''),
                    num=(r.num if r else ()), seps=(r.seps if r else ()),
                    pada=(r.pada if r else ''), speaker=None, meter=None, syllables=0,
                    padas=[], **kw)

def _units(buf, page, st, scheme, meter):
    """Split one buffered block on the double daṇḍa and attach the references that sit between them.

    `st` carries the two things that straddle a block boundary: a reference that appeared on its own
    line before the verse, and the speaker currently holding the floor."""
    out, parts = L(), _DANDA2.split(buf)
    # `(text, closing daṇḍa)` pairs: the delimiter is kept with the unit it closes, so the stored
    # verse reads the way it is printed instead of losing its last mark to the splitter
    for i in range(0, len(parts), 2):
        p, dan = parts[i].strip(' \t\r\n'), (parts[i+1] if i+1 < len(parts) else '')
        if not p: continue
        if (r := parse_ref(p)) is not None and (len(r.num) > 1 or r.siglum or out):
            # a token that is only a reference belongs to the unit it follows
            if out and not out[-1].ref:
                u = out[-1]
                u.ref, u.siglum, u.num, u.seps, u.pada = p.translate(DEV_DIGITS), r.siglum, r.num, r.seps, r.pada
            elif not out: st['ref'] = p.translate(DEV_DIGITS)
            continue
        u = _mk('verse', f'{p} {dan}'.strip(), page, st.pop('ref', None))
        u.padas = [x.strip() for x in _DANDA1.split(p) if x.strip()]
        # `arjuna uvāca` stays in the text — it is what a reader searches for — but it is a stage
        # direction, not a pāda, and scanning it makes every line of dialogue read as prose
        body = []
        for ln in p.splitlines():
            if (m := _SPEAKER.match(ln.strip())): st['speaker'] = m.group(1).strip()
            else: body.append(ln)
        u.speaker = st.get('speaker')
        if meter:
            scan = '\n'.join(body)
            w = [g for _, g in syllables(scan, scheme)]
            u.syllables = len(w)
            m = detect_meter(scan, scheme, w)
            u.meter = m.name if m else None
            if not is_verse(scan, scheme, w): u.kind = 'prose'
        out.append(u)
    return out

def split_verses(text,                  # one Sanskrit e-text: GRETIL plain text, Devanagari, SLP1, ...
                 page:int=0,            # page number stamped on every unit from this text
                 header:bool=True,      # drop a GRETIL-style front matter block first
                 scheme:str=None,       # transliteration of the source (None -> Devanagari auto-detected)
                 meter:bool=True        # scan each unit, which also classifies verse vs prose
) -> L:
    """The units of a Sanskrit text, one record per verse.

    **The verse is the atom.** Half a śloka is not a proposition — it is a metrical fragment whose
    embedding is noise and whose text answers nothing — so nothing in this module ever cuts inside
    one. That makes segmentation the whole game, and the good news is that Sanskrit e-texts mark it
    explicitly: a single daṇḍa closes a hemistich, a double daṇḍa closes the verse, and GRETIL puts
    the citation between two more (`... abruvan // Mn_1.1 //`).

    Four layouts are handled, because the corpora disagree:

    | layout | example | seen in |
    |---|---|---|
    | reference after the verse | `... abruvan // Mn_1.1 //` | most of GRETIL |
    | reference on its own line | `BhG 3.1` then the verse, closing `|| 1 ||` | GRETIL commentary files |
    | Devanagari with a bare number | `... सञ्जय ॥ १ ॥` | most Devanagari editions |
    | unmetred prose between daṇḍas | Vedic prose, sūtra, bhāṣya | Saṃhitās, commentaries |

    Colophons (`iti ... adhyāyaḥ`), GRETIL's `[h: :h]` headings and speaker lines (`X uvāca`) come
    back as their own records, because `verse_tree` needs them to find sections in the many texts
    that have no headings at all."""
    txt = _PAGEMK.sub(' ', strip_header(text) if header else (text or '')).replace('§', '')
    txt = _GKOLO.sub(lambda m: f'\n[[{m.group(1)}]]\n', _GHEAD.sub(lambda m: f'\n[h]{m.group(1)}\n', txt))
    out, buf, st = L(), [], {}
    def flush():
        if (b := '\n'.join(buf).strip()): out.extend(_units(b, page, st, scheme, meter))
        buf.clear()
    for ln in txt.splitlines():
        s = ln.strip()
        if not s: buf.append(''); continue
        if s.startswith('[h]'):
            flush(); out.append(_mk('heading', s[3:], page)); continue
        if (m := _COLOPHON.match(s)) and len(s) < 240:
            flush(); out.append(_mk('colophon', m.group(1), page)); continue
        # A division heading is checked *before* a reference, because `Sarga 3` parses as both and
        # only one of them is right: a citation siglum is `Mn`, `ViP`, `BhG` — never a division word.
        if unit_heading(s) and not _BODY_LINE.search(s):
            flush(); out.append(_mk('heading', s, page)); continue
        # a bare reference on its own line labels the verse that follows it
        if (r := parse_ref(s)) is not None and (r.siglum or len(r.num) > 1):
            flush(); st['ref'] = s.translate(DEV_DIGITS); continue
        buf.append(s)
    flush()
    for i, u in enumerate(out): u.pos = i
    return out

def split_commentary(text, min_gloss:int=40):
    """A mūla-plus-commentary block split into `(mula, [(author, gloss), ...])`.

    GRETIL's commentary files introduce each commentator by name and a colon on a line of its own
    (`śrīdharaḥ :`), which is the only marker there is — there is no markup. Keeping the pairing is
    what lets a chunk carry the verse *and* one commentator's reading of it: the question is
    usually answered by the gloss, but it is the verse that the reader recognises and searches for."""
    lns, cuts = (text or '').splitlines(), []
    for i, ln in enumerate(lns):
        s = ln.strip()
        if 4 <= len(s) <= 60 and s.endswith(':') and not _DANDA1.search(s) and len(s.split()) <= 4:
            cuts.append((i, s.rstrip(':').strip()))
    if not cuts: return (text or '').strip(), []
    mula, out = '\n'.join(lns[:cuts[0][0]]).strip(), []
    for j, (i, who) in enumerate(cuts):
        end = cuts[j+1][0] if j+1 < len(cuts) else len(lns)
        body = '\n'.join(lns[i+1:end]).strip()
        if len(body) >= min_gloss: out.append((who, body))
    return mula, out

In [ ]:
MANU = '''  Manu-Smrti (plain text)

Typed, analyzed and proofread by M.YANO and Y.IKARI
For a complete list of GRETIL encodings see http://gretil.sub.uni-goettingen.de/gretdiac.pdf

manum ekāgram āsīnam abhigamya maharṣayaḥ /
pratipūjya yathānyāyam idaṃ vacanam abruvan // Mn_1.1 //
bhagavan sarvavarṇānāṃ yathāvad anupūrvaśaḥ /
antaraprabhavānāṃ ca dharmān no vaktum arhasi // Mn_1.2 //'''
vs = split_verses(MANU)
assert len(vs) == 2, len(vs)                      # the bibliography did not become a verse
assert [v.ref for v in vs] == ['Mn_1.1', 'Mn_1.2']
assert vs[0].num == (1, 1) and vs[0].siglum == 'Mn' and vs[0].meter == 'anuṣṭubh'
assert len(vs[0].padas) == 2 and vs[0].text.startswith('manum')

# the same verse in Devanagari with a bare number, and the fold that makes it searchable
BHG = '''धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः ।
मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय ॥ १ ॥'''
v = split_verses(BHG, header=False)[0]
assert v.ref == '1' and v.meter == 'anuṣṭubh' and v.syllables == 32
assert 'dharmaksetre kuruksetre' in fold(v.text)

# a reference on its own line, the layout of GRETIL's commentary files
v = split_verses('BhG 3.1\n\narjuna uvāca\njyāyasī cet karmaṇas te matā buddhir janārdana |\n'
                 'tat kiṃ karmaṇi ghore māṃ niyojayasi keśava || 1 ||', header=False)[0]
assert v.num == (3, 1) and v.speaker == 'arjuna'
# `arjuna uvāca` is kept in the text but kept out of the scan
assert 'arjuna uvāca' in v.text and v.meter == 'anuṣṭubh'


# Header stripping must not eat the opening pāda. Only the *hemistich* carries a daṇḍa, so in a
# four-line metre line 1 has no mark on it -- and a verse missing its first quarter does not scan.
MD_FILE = """Kalidasa: Meghaduta
PLAIN TEXT VERSION
For a list of GRETIL encodings see http://gretil.sub.uni-goettingen.de/gretdiac.pdf

kaścit kāntāvirahaguruṇā svādhikārātpramattaḥ
śāpenāstaṃgamitamahimā varṣabhogyeṇa bhartuḥ /
yakṣaścakre janakatanayāsnānapuṇyodakeṣu
snigdhacchāyātaruṣu vasatiṃ rāmagiryāśrameṣu // MD_1.1 //"""
v = split_verses(MD_FILE)[0]
assert v.text.startswith('kaścit') and v.meter == 'mandākrāntā' and v.syllables == 68
assert 'Kalidasa' not in v.text and 'PLAIN TEXT' not in v.text

In [ ]:
# colophons, speakers, page markers and Vedic prose all come back correctly labelled
VIP = '''parāśaraṃ munivaraṃ kṛtapūrvāhṇikakriyam /
maitreyaḥ paripapraccha praṇipatyābhivādya ca // ViP_1,1.1 //
[[iti śrīviṣṇupurāṇe prathame 'ṃśe prathamo 'dhyāyaḥ ]]
[Page I,12]
maitreya uvāca
kathaṃ sṛṣṭaṃ jagat pūrvaṃ kathaṃ ca pralayaṃ gatam /
kiṃ ca tasya jagat kartā tan me brahman prakīrtaya // ViP_1,2.1 //
goṣad asi pratyuṣṭaṃ rakṣaḥ pratyuṣṭārātiḥ //'''
vs = split_verses(VIP, header=False)
assert [v.kind for v in vs] == ['verse', 'colophon', 'verse', 'prose']
assert vs[2].speaker == 'maitreya' and vs[0].speaker is None
assert '[Page' not in vs[2].text                       # the page marker was dropped, not indexed
assert vs[1].text.startswith('iti śrīviṣṇupurāṇe')
assert vs[3].kind == 'prose'                           # unmetred Vedic prose is not a śloka

# the colophon names the whole ladder it closes -- which is how the tree levels get their names
assert unit_words("iti śrīviṣṇupurāṇe prathame 'ṃśe prathamo 'dhyāyaḥ") == ['amsa', 'adhyaya']
assert unit_words('iti śrīmadrāmāyaṇe bālakāṇḍe prathamaḥ sargaḥ') == ['kanda', 'sarga']
assert unit_words("iti śrīmahābhārate ādiparvaṇi prathamo 'dhyāyaḥ") == ['parva', 'adhyaya']
# ... and a division word needs a number, or every philosophical verse becomes a chapter break
assert unit_heading('Sarga 3').unit == 'sarga' and unit_heading('॥ अध्याय ३ ॥').n == 3
assert unit_heading("atha tṛtīyo 'dhyāyaḥ").n == 3
assert unit_heading('padārthaḥ kaścit') is None and _unit_rank('padma') is None

# `Sarga 3` parses as a reference too (siglum `Sarga`, number 3); the heading reading wins, because
# no citation siglum is ever a division word
assert split_verses('Sarga 3\nkiṃ cid uktam //', header=False)[0].kind == 'heading'
assert split_verses('BhG 3.1\nkiṃ cid uktam //', header=False)[0].ref == 'BhG 3.1'

assert parse_ref('ViP_1,1.1').seps == (',', '.') and format_ref('ViP', (1, 1), (',',)) == 'ViP_1,1'
assert parse_ref('MBh_1,1.1').num == (1, 1, 1) and parse_ref('not a ref') is None

## Chunking

`nbs/07_doc_eval.ipynb` found that chunk granularity dominated every structural feature it tested —
finer chunks alone moved MRR from 0.330 to 0.522. Sanskrit sits at the *other* end of that curve:
a śloka is about 32 syllables, roughly 120 characters, and a chunk that small starves both legs of
the search. BM25 has almost no term statistics to weigh, and a dense vector over a dozen words is
dominated by whichever one is rarest.

So the unit here is a **window of whole verses**, overlapping. `per` sets the size, `stride` the
overlap, and `max_chars` shrinks the window rather than ever splitting a verse.

`sanskrit_chunks` then chunks verse and prose runs *separately*, because a Sanskrit text is usually
both — mantra interleaved with Vedic prose, sūtra with bhāṣya, verse with gloss — and one splitter
gets one of the two wrong.

In [ ]:
#| export
def chunk_verses(verses,               # verse records from `split_verses`
                 per:int=4,            # verses per chunk
                 stride:int=None,      # verses to advance between chunks (default `per-1`)
                 max_chars:int=1600,   # shrink the window rather than exceed this
                 sep:str='\n'
) -> L:
    """Verse-atomic windows over a run of verses.

    A śloka is about 32 syllables — 120 characters, a dozen words. That is well under the size at
    which either leg of a hybrid search works: BM25 has almost no term statistics to go on and a
    dense vector of twelve words is dominated by whichever word is rarest. Windowing several verses
    restores a workable chunk **without ever cutting inside one**, and `stride < per` overlaps the
    windows so a thought that runs across a verse boundary sits whole inside at least one chunk.

    `max_chars` shrinks the window instead of splitting a verse: the invariant is that every chunk
    is a whole number of verses, and it is the one thing here that is never traded away."""
    vs = [v for v in L(verses) if (v.get('text') or '').strip()]
    if not vs: return L()
    per = max(1, per); stride = max(1, ifnone(stride, max(1, per-1)))
    out, i = L(), 0
    while i < len(vs):
        win = vs[i:i+per]
        while len(win) > 1 and sum(len(v['text']) for v in win) > max_chars: win = win[:-1]
        refs = [v['ref'] for v in win if v.get('ref')]
        out.append(dict(content=sep.join(v['text'] for v in win), verses=win,
                        refs=refs, ref=first(refs, lambda r: True) or '', ref_end=(refs[-1] if refs else ''),
                        nverses=len(win), page=win[0].get('page', 0),
                        meters=list(dict.fromkeys(v['meter'] for v in win if v.get('meter'))),
                        speaker=win[0].get('speaker'), kind=win[0].get('kind', 'verse'),
                        lemmas=' '.join(v.get('lemmas') or '' for v in win).strip()))
        if i + len(win) >= len(vs): break
        i += min(stride, len(win))
    return out

def _runs(verses):
    'Consecutive verses grouped into runs of one kind, so verse and bhāṣya can be chunked differently.'
    out = L()
    for v in verses:
        k = 'prose' if v.get('kind') == 'prose' else 'verse'
        if out and out[-1][0] == k: out[-1][1].append(v)
        else: out.append((k, [v]))
    return out

def _chunk_meta(c, translit=True, lemmas=None):
    'The JSON that rides in the store\'s `metadata` column — and is therefore FTS-indexed for free.'
    m = dict(ref=c.get('ref',''), ref_end=c.get('ref_end',''),
             refs=' ' + ' '.join(c.get('refs') or []) + ' ',      # padded so LIKE can match exactly
             nverses=c.get('nverses', 1), kind=c.get('kind','verse'))
    if c.get('meters'):  m['meter'] = ' '.join(c['meters'])
    if c.get('on'):      m['on'] = c['on']      # this chunk glosses that verse, it is not that verse
    if c.get('speaker'): m['speaker'] = c['speaker']
    if translit:         m['fold'] = fold(c['content'])
    if c.get('lemmas'):  m['lemmas'] = fold(c['lemmas'])
    if lemmas:           m['lemmas'] = ((m.get('lemmas','') + ' ' + fold(lemmas)).strip())
    return m

def sanskrit_chunks(tree,                  # nodes from `verse_tree`
                    title:str,             # document title
                    did:str,               # document id
                    chunker=None,          # chonkie chunker for the prose runs (bhāṣya, Vedic prose)
                    per:int=4,             # verses per chunk
                    inherit_ref:bool=True, # an unreferenced prose run is a gloss on the verse above it
                    stride:int=None,       # verses advanced between chunks
                    max_chars:int=1600,
                    min_chunk:int=MIN_CHUNK,
                    translit:bool=True,    # store the fold of each chunk for scheme-agnostic FTS
                    lemma_fn=None          # list[str] -> list[str]; a segmenter/lemmatiser seam
) -> L:
    """Chunk a verse tree: verse windows for the verse, an ordinary chunker for the prose.

    A Sanskrit text is usually both. The Saṃhitās interleave mantra and Vedic prose; a śāstra
    alternates sūtra with bhāṣya; a commentary file is verse, gloss, verse, gloss. Sending all of it
    through one splitter gets one of the two wrong, so the runs are chunked separately and put back
    in document order.

    Each chunk carries its citation range in `metadata`, plus `fold` — the diacritic-free skeleton
    that makes `get_store`'s existing FTS index match a query typed in any transliteration. That is
    the whole reason this module needs no schema of its own.

    `lemma_fn` is the seam for real linguistics. Nothing here needs it: pass the ByT5-Sanskrit
    segmenter, or lemmas already carried by a DCS CoNLL-U file, and they are indexed beside the
    surface text — which matters more in Sanskrit than in any other language, because sandhi means
    the surface form of a word is frequently not the form anyone will type."""
    out = L()
    for nd in tree:
        head, nid = heading_path(tree, nd, title), f'{did}#{nd.seq}'
        cs, mula = L(), ''
        for kind, run in _runs(nd.meta.get('verses') or []):
            if kind == 'verse':
                vc = chunk_verses(run, per, stride, max_chars)
                cs += vc
                mula = first((c['ref'] for c in reversed(vc) if c['ref']), mula)
            else:
                for v in run:
                    # A bhāṣya block carries no citation of its own; it is a gloss on the verse it
                    # follows. Inheriting that citation is what makes `by_ref('BhG_3.1')` return the
                    # verse *and* its commentaries, which is how the text is actually read.
                    r = v.get('ref') or (mula if inherit_ref else '')
                    for t in (chunk_markdown(v['text'], chunker) if len(v['text']) > max_chars else [v['text']]):
                        if (t or '').strip():
                            cs.append(dict(content=t, verses=[v], refs=[r] if r else [],
                                           ref=r, ref_end=r, nverses=1, on=('' if v.get('ref') else mula),
                                           page=v.get('page',0), meters=[], speaker=v.get('speaker'),
                                           kind='prose', lemmas=v.get('lemmas') or ''))
        # A chunk under `min_chunk` merges into its predecessor, as in `tree.node_chunks` — but the
        # merge happens *before* the metadata is built, so it keeps both windows' citations and
        # lemmas. Merging the text and not the metadata is the version of this that silently loses
        # a verse's address: the text is still searchable, `by_ref` can no longer find it.
        merged = L()
        for c in cs:
            if merged and len(c['content'].strip()) < min_chunk:
                m = merged[-1]
                m['content'] += '\n' + c['content']
                for k in ('verses', 'refs'): m[k] = list(m[k]) + list(c[k])
                m['meters']  = list(dict.fromkeys(list(m['meters']) + list(c['meters'])))
                m['ref_end'] = c['ref_end'] or m['ref_end']
                m['on'] = m.get('on') or c.get('on', '')
                m['nverses'] += c['nverses']
                m['lemmas']  = f"{m.get('lemmas','')} {c.get('lemmas','')}".strip()
                continue
            merged.append(dict(c))
        lms = lemma_fn([c['content'] for c in merged]) if (lemma_fn and merged) else [None]*len(merged)
        for c, lm in zip(merged, lms):
            md = _chunk_meta(c, translit, lm)
            bc = f"{head} › {md['ref']}" if md['ref'] else head
            out.append(dict(content=c['content'], doc_id=did, node_id=nid, page=c['page'],
                            heading=bc[:200], metadata=json.dumps(md, ensure_ascii=False)))
    return out

In [ ]:
vs = split_verses('\n'.join(f'tatra prathamam idam uktam {i} /\ntato dvitīyam idam uktam {i} // T_1.{i} //'
                            for i in range(1, 8)), header=False)
assert len(vs) == 7
cs = chunk_verses(vs, per=3, stride=2)
# every chunk is a whole number of verses, and the overlap covers every boundary
assert all(c['nverses'] <= 3 for c in cs) and all(c['content'].count('//') == c['nverses'] for c in cs)
assert [c['ref'] for c in cs] == ['T_1.1', 'T_1.3', 'T_1.5'] and cs[-1]['ref_end'] == 'T_1.7'
assert {v['ref'] for c in cs for v in c['verses']} == {f'T_1.{i}' for i in range(1, 8)}  # nothing lost

# max_chars shrinks the window; it never splits a verse
tight = chunk_verses(vs, per=4, max_chars=90)
assert all(c['content'].count('//') == c['nverses'] for c in tight)
assert max(c['nverses'] for c in tight) < 4

`split_commentary` is the other half of the mūla/bhāṣya story: it recovers the pairing
from the only marker GRETIL gives it, a commentator's name and a bare colon on a line of
its own. Keeping the pair together is what lets a chunk carry the verse *and* one reading of
it — the question is usually answered by the gloss, but it is the verse the reader recognises.

In [ ]:
# a mula verse with two commentaries, the layout of GRETIL's `*4c*` files
COMM = '''BhG 3.1

arjuna uvāca
jyāyasī cet karmaṇas te matā buddhir janārdana |
tat kiṃ karmaṇi ghore māṃ niyojayasi keśava || 1 ||

śrīdharaḥ :

evaṃ tāvad aśocyān anvaśocas tvam ity ādinā prathamaṃ mokṣa-sādhanatvena
dehātma-viveka-buddhir uktā | tad-anantaram eṣā te 'bhihitā sāṅkhye buddhir |

madhusūdanaḥ :

karma-yogasya jñāna-yogāpekṣayā śraiṣṭhyaṃ pratipādayiṣyan bhagavān
arjunasya praśnam avatārayati | jyāyasī cet iti |
'''
mula, glosses = split_commentary(COMM)
assert [w for w, _ in glosses] == ['śrīdharaḥ', 'madhusūdanaḥ']
assert 'jyāyasī cet karmaṇas te' in mula
assert all('|' in g for _, g in glosses)
print(mula.strip()[:60], '||', [(w, g[:40]) for w, g in glosses])

## Sections without headings

This is where a Sanskrit corpus is *easier* than a PDF, not harder. `tree.detect_mode` has to infer
a hierarchy from words on the page and can be fooled — the evaluation notebook records a 5,242-node
"tree" of one-line sections built from a converter's stray `h1`s. A referenced Sanskrit text needs
no inference: `ViP_1,1.1` states a complete address, for every leaf, exactly.

`verse_mode` therefore tries the reference first, then the colophons, then unit headings, then fixed
verse windows — a different order from `detect_mode`, for a corpus with different strengths.

Two small things that turned out to matter more than expected:

- **Node titles are citations.** `ViP_1,1` is what a Sanskritist writes, and it makes a breadcrumb
  unambiguous in a way `Section 3` never is. When the colophons name the ladder, the levels get
  those names instead (`Amsa 1 › Adhyaya 1`).
- **The colophon becomes the node summary.** It is a one-line statement of what the section was,
  written by the tradition itself, and it beats the first 300 characters that
  `summarize_extractive` would otherwise hand `toc()`.

In [ ]:
#| export
def verse_mode(verses, min_ref_frac:float=0.6) -> str:
    """Which structural signal a Sanskrit text carries: `ref`, `colophon`, `heading` or `window`.

    The order is not the same as `tree.detect_mode`'s and that is the point of this module. A
    printed book puts its structure in its headings; a Sanskrit e-text usually has **no headings at
    all** and puts its structure in the citation stamped on every single verse. `ViP_1,1.1` is a
    complete address — aṃśa 1, adhyāya 1, verse 1 — so where prose has to infer a hierarchy from
    the words on the page, a referenced text simply states one, for every leaf, exactly."""
    body = [v for v in verses if v.kind in ('verse','prose')]
    if not body: return 'window'
    if len([v for v in body if len(v.num) >= 2])/len(body) >= min_ref_frac: return 'ref'
    if sum(1 for v in verses if v.kind == 'colophon') >= 2: return 'colophon'
    if sum(1 for v in verses if v.kind == 'heading') >= 2: return 'heading'
    return 'window'

def infer_units(verses, depth:int) -> list:
    'The names for `depth` tree levels, read off the colophons, or `[]` when they do not agree.'
    seqs = [w for v in verses if v.kind in ('colophon','heading') and len(w := unit_words(v.text)) == depth]
    if not seqs: return []
    top = Counter(tuple(s) for s in seqs).most_common(1)[0]
    return list(top[0]) if top[1] >= max(1, len(seqs)//2) else []

def verse_tree(pages,                   # [(page_no, text)] — or the output of `add_sanskrit_file`
               title:str='Document',
               summarize=None,          # callable(text)->str for node summaries
               units=None,              # level names, e.g. 'kanda,sarga' (None -> read off the colophons)
               max_levels:int=4,        # deepest node level kept
               window:int=40,           # verses per node when the text carries no structure at all
               min_ref_frac:float=0.6,  # fraction of verses that must be referenced to trust the refs
               scheme:str=None,         # transliteration of the source
               header:bool=True,        # drop a GRETIL front matter block
               mode:str=None,           # force a mode instead of asking `verse_mode`
               verses=None              # pre-split verse records, if you already have them
) -> list:
    """A `TreeNode` list for a Sanskrit text, built from its verse references.

    Node titles are **citations** by default — `ViP_1`, `ViP_1,1` — because that is the address a
    Sanskritist actually writes, and it makes a breadcrumb unambiguous in a way `Section 3` never
    is. When the colophons name the ladder (`iti ... prathame 'mse prathamo 'dhyayah`) the levels
    are named after it instead: `Amsa 1 › Adhyaya 1`. Pass `units=` to override both.

    The colophon does more work than naming: it becomes the node's **summary**. A colophon is a
    one-line statement of what the section was, written by the tradition itself, and it beats the
    first 300 characters of the section that `summarize_extractive` would otherwise hand `toc()`.

    Every node keeps its verse records in `TreeNode.meta['verses']`, which is what `sanskrit_chunks`
    windows. That is the whole contract between the two halves of this module, and it is why both
    plug into `db.add_doc` through its `tree_fn` / `chunk_fn` arguments with nothing else changed."""
    pages = [(p, t or '') for p, t in pages]
    vs = L(verses) if verses is not None else L(v for p, t in pages
                                                for v in split_verses(t, page=p, header=header, scheme=scheme))
    if not vs: return build_tree(pages, title=title, summarize=summarize)
    body = vs.filter(lambda v: v.kind in ('verse','prose'))
    if not body: return build_tree(pages, title=title, summarize=summarize)
    # a single-page source (one GRETIL file) has no pages to speak of, so the verse ordinal is the
    # honest position: it is what `toc()` shows and what `merge_spans` measures adjacency in
    if len({p for p, _ in pages}) <= 1:
        for i, v in enumerate(body): v.page = i
    mode = ifnone(mode, verse_mode(vs, min_ref_frac))
    summarize, nodes = summarize or summarize_extractive, []
    def fresh(t, lvl, parent, page):
        nd = TreeNode(seq=len(nodes), title=_clean_title(t) or f'Section {len(nodes)}', level=lvl,
                      parent=parent, page_start=page, page_end=page)
        nodes.append(nd)
        if parent is not None: nodes[parent].children.append(nd.seq)
        return nd
    root = fresh(title, 0, None, first(body).page)
    def assign(nd, v):
        nd.meta.setdefault('verses', []).append(v)
        nd.segments.append((v.page, v.text))
        nd.page_end = max(nd.page_end, v.page)
    last = [root]

    if mode == 'ref':
        deep  = body.filter(lambda v: len(v.num) >= 2)
        depth = min(Counter(len(v.num) for v in deep).most_common(1)[0][0] - 1, max_levels)
        names = (L(units.split(',') if isinstance(units, str) else units).map(str.strip)
                 if units else L(infer_units(vs, depth)))
        cache = {}
        for v in vs:
            if v.kind in ('colophon','heading'):
                last[0].meta.setdefault('colophon', v.text); continue
            parent = root
            for d in range(1, min(depth, len(v.num))+1):
                key = v.num[:d]
                if key not in cache:
                    nm = (f'{names[d-1].title()} {v.num[d-1]}' if d-1 < len(names)
                          else format_ref(v.siglum, key, v.seps))
                    cache[key] = fresh(nm, d, parent.seq, v.page)
                parent = cache[key]
            assign(parent, v); last[0] = parent
    elif mode == 'colophon':
        cur = [None]
        def node():
            if cur[0] is None: cur[0] = fresh(f'Section {len(nodes)}', 1, root.seq, last[0].page_end)
            return cur[0]
        for v in vs:
            if v.kind == 'colophon':
                nd = node(); nd.title = _clean_title(v.text, 100); nd.meta['colophon'] = v.text
                cur[0] = None; continue
            if v.kind == 'heading': continue
            assign(node(), v); last[0] = node()
    elif mode == 'heading':
        heads = L(vs).filter(lambda v: v.kind == 'heading').map(lambda v: unit_heading(v.text)).filter(None)
        order = sorted({h.level for h in heads})
        cur = [root]
        for v in vs:
            if v.kind == 'heading':
                h = unit_heading(v.text)
                lvl = min(order.index(h.level)+1, max_levels) if h else 1
                while len(nodes) and cur[0].level >= lvl and cur[0].parent is not None:
                    cur[0] = nodes[cur[0].parent]
                cur[0] = fresh(v.text, lvl, cur[0].seq, v.page); continue
            if v.kind == 'colophon': cur[0].meta.setdefault('colophon', v.text); continue
            assign(cur[0], v); last[0] = cur[0]
    else:
        for i in range(0, len(body), window):
            grp = body[i:i+window]
            nd = fresh(f'Verses {i+1}–{i+len(grp)}', 1, root.seq, grp[0].page)
            for v in grp: assign(nd, v)
    for nd in reversed(nodes):     # a parent covers its children, so `toc()` shows real ranges
        if nd.parent is not None:
            up = nodes[nd.parent]
            up.page_start = min(up.page_start, nd.page_start) if up.segments or up.children else nd.page_start
            up.page_end = max(up.page_end, nd.page_end)
    for nd in nodes:
        base = nd.meta.get('colophon') or nd.text() or ' / '.join(nodes[c].title for c in nd.children[:8])
        nd.summary = summarize(base) if base else ''
    root.page_end = max((n.page_end for n in nodes), default=root.page_end)
    return nodes

In [ ]:
VIP = '''parāśaraṃ munivaraṃ kṛtapūrvāhṇikakriyam /
maitreyaḥ paripapraccha praṇipatyābhivādya ca // ViP_1,1.1 //
tvatto hi vedādhyayanam adhītam akhilaṃ guro /
dharmaśāstrāṇi sarvāṇi vedāṅgāni yathākramam // ViP_1,1.2 //
[[iti śrīviṣṇupurāṇe prathame 'ṃśe prathamo 'dhyāyaḥ ]]
kathaṃ sṛṣṭaṃ jagat pūrvaṃ kathaṃ ca pralayaṃ gatam /
kiṃ ca tasya jagat kartā tan me brahman prakīrtaya // ViP_1,2.1 //
brahmādyaṃ sarvabhūtānāṃ prabhavaṃ ca nibodha me /
sarvasya jagato yonir viṣṇur nārāyaṇaḥ paraḥ // ViP_1,2.2 //
[[iti śrīviṣṇupurāṇe prathame 'ṃśe dvitīyo 'dhyāyaḥ ]]
ādyo 'yaṃ dvitīyaḥ sargaḥ kathitas te mayānagha /
tṛtīyaṃ śṛṇu me tāta yathā sṛṣṭaṃ jagat punaḥ // ViP_2,1.1 //'''
assert verse_mode(split_verses(VIP, header=False)) == 'ref'
t = verse_tree([(0, VIP)], title='Visnu-Purana', header=False)
# the ladder is named from the colophons, not numbered generically
assert [(n.level, n.title) for n in t] == [
    (0,'Visnu-Purana'), (1,'Amsa 1'), (2,'Adhyaya 1'), (2,'Adhyaya 2'), (1,'Amsa 2'), (2,'Adhyaya 1')]
# a parent covers its children, and the colophon became the section summary
assert (t[1].page_start, t[1].page_end) == (0, 3)
assert t[2].summary.startswith('iti śrīviṣṇupurāṇe') and t[2].meta['verses'][0].ref == 'ViP_1,1.1'
# without colophons the titles fall back to citations, which is what a Sanskritist writes
bare = verse_tree([(0, '\n'.join(l for l in VIP.splitlines() if not l.startswith('[[')))],
                  title='V', header=False)
assert [n.title for n in bare if n.level] == ['ViP_1', 'ViP_1,1', 'ViP_1,2', 'ViP_2', 'ViP_2,1']

# no references at all -> fixed verse windows, so every text is still navigable
plain = '\n'.join(f'tatra prathamam idam uktam {i} /\ntato dvitiyam idam uktam {i} //' for i in range(12))
w = verse_tree([(0, plain)], title='W', header=False, window=5)
assert verse_mode(split_verses(plain, header=False)) == 'window'
assert [n.title for n in w if n.level] == ['Verses 1–5', 'Verses 6–10', 'Verses 11–12']


# `colophon` mode: no references anywhere, but the tradition marks each section's end -- and names it
COLO = '\n'.join(
    [f'tatra prathamam idam uktam {i} /\ntato dvitiyam idam uktam {i} //' for i in range(3)]
    + ["[[iti śrīmadgrانthe prathamo 'dhyāyaḥ ]]".replace('ا', 'a')]
    + [f'tatra prathamam idam uktam 1{i} /\ntato dvitiyam idam uktam 1{i} //' for i in range(2)]
    + ["[[iti śrīmadgranthe dvitīyo 'dhyāyaḥ ]]"])
assert verse_mode(split_verses(COLO, header=False)) == 'colophon'
ct = verse_tree([(0, COLO)], title='C', header=False)
assert [n.level for n in ct] == [0, 1, 1]
assert all(n.title.startswith('iti') for n in ct if n.level) and len(ct[1].meta['verses']) == 3

# `heading` mode: unit headings in the text, compacted onto consecutive levels
HEAD = """Kanda 1
Sarga 1
tatra prathamam idam uktam /
tato dvitiyam idam uktam //
Sarga 2
tatra prathamam idam uktam 2 /
tato dvitiyam idam uktam 2 //
Kanda 2
Sarga 1
tatra prathamam idam uktam 3 /
tato dvitiyam idam uktam 3 //"""
assert verse_mode(split_verses(HEAD, header=False)) == 'heading'
ht = verse_tree([(0, HEAD)], title='H', header=False)
assert [(n.level, n.title) for n in ht if n.level] == [
    (1,'Kanda 1'), (2,'Sarga 1'), (2,'Sarga 2'), (1,'Kanda 2'), (2,'Sarga 1')]

## Ingestion and search

`add_sanskrit` is `add_doc` with its two seams filled in. `add_sanskrit_file` reads GRETIL's `.htm`
plain-text editions directly (the markup is stripped, not parsed — there is none beyond `<br>`).

On the query side, `verse_search` supports `emb=None` as a **first-class mode** rather than a
degradation: looking up a half-remembered verse is a lexical act, and the fold makes it work across
scripts with no model in the path at all. `by_ref` is the exact lookup — `db.by_ref('ViP_1,1.1')`.

In [ ]:
#| export
SANSKRIT_EXTS = '.txt,.htm,.html,.md,.iast,.slp1,.hk'

_TAGS  = re.compile(r'<[^>]+>')
_BLOCK = re.compile(r'</?(?:br|p|div|tr|li|h[1-6])\b[^>]*>', re.I)
_DROP  = re.compile(r'<(script|style|head)\b.*?</\1>', re.S|re.I)

def read_text(path, scheme:str=None) -> str:
    """One Sanskrit e-text as plain text. GRETIL serves its plain-text editions as `.htm`, so the
    markup is stripped rather than parsed — there is none to speak of beyond `<br>` and `<p>`."""
    p = Path(path)
    t = p.read_text(encoding='utf-8', errors='replace')
    if p.suffix.lower() in ('.htm','.html','.xml'):
        from html import unescape
        t = unescape(_TAGS.sub('', _BLOCK.sub('\n', _DROP.sub(' ', t))))
    return t

def sanskrit_embed(emb_fn, mode:str='as-is'):
    """Wrap an embedder so every chunk reaches it in one transliteration: `as-is`, `iast` or `fold`.

    Left at `as-is` by default and deliberately not sold as an improvement. The argument for `iast`
    is that a multilingual model has seen far more romanised Sanskrit than Devanagari, so
    normalising a mixed corpus onto one script should help; the argument against is that folding
    away `ś`/`ṣ`/`s` destroys distinctions the tokeniser may well be using. **Neither is measured
    here.** It is a knob with an honest label, not a recommendation — run `nbs/07_doc_eval.ipynb`
    over your own corpus before turning it on."""
    if not emb_fn or mode == 'as-is': return emb_fn
    tf = to_iast if mode == 'iast' else fold
    def _(txts, **kw): return emb_fn([tf(t) for t in txts], **kw)
    return _

@patch
def add_sanskrit(self:Database,
                 pages,                 # [(page_no, text)] — or a single string
                 title:str,             # document title
                 source:str=None,       # path or url (defaults to the title)
                 kind:str='sanskrit',
                 store:str='store',
                 prefix:str=None,
                 emb_fn=None,           # embedder: list[str] -> vectors
                 per:int=4,             # verses per chunk
                 stride:int=None,       # verses advanced between chunks (default `per-1`)
                 max_chars:int=1600,
                 units=None,            # tree level names, e.g. 'kanda,sarga'
                 scheme:str=None,       # transliteration of the source
                 header:bool=True,      # drop a GRETIL front matter block
                 emb_text:str='as-is',  # transliteration handed to the embedder (see `sanskrit_embed`)
                 lemma_fn=None,         # list[str] -> list[str]; segmenter/lemmatiser seam
                 translit:bool=True,    # store each chunk's fold for scheme-agnostic FTS
                 min_chunk:int=MIN_CHUNK,  # chunks shorter than this merge into their predecessor
                 chunker=None,          # chonkie chunker for the prose runs
                 verses=None,           # pre-split verse records (see `add_dcs`)
                 **kw                   # forwarded to `add_doc` (summarize, meta, force, ...)
) -> dict:
    """Ingest one Sanskrit text: reference tree, verse-window chunks, scheme-folded FTS keys.

    This is `db.add_doc` with its two seams filled in — `tree_fn=verse_tree` and
    `chunk_fn=sanskrit_chunks` — so everything downstream is the machinery that was already there.
    `toc`, `read`, `breadcrumb`, `doc_search`, `sections`, `get_graph` and `graph_search` all work
    on the result with no Sanskrit-specific code path at all."""
    tf = lambda pgs, title, summarize=None: verse_tree(pgs, title=title, summarize=summarize,
                                                      units=units, scheme=scheme, header=header,
                                                      verses=verses)
    cf = lambda tree, title, did, chunker=None: sanskrit_chunks(tree, title, did, chunker, per=per,
        stride=stride, max_chars=max_chars, translit=translit, lemma_fn=lemma_fn, min_chunk=min_chunk)
    return self.add_doc(pages, title, source=source, kind=kind, store=store, prefix=prefix,
                        emb_fn=sanskrit_embed(emb_fn, emb_text), chunker=chunker,
                        tree_fn=tf, chunk_fn=cf, **kw)

@patch
def add_sanskrit_file(self:Database, path, title:str=None, store:str='store', prefix:str=None, **kw) -> dict:
    'Ingest one Sanskrit e-text file (GRETIL `.htm`, `.txt`, Devanagari or romanised).'
    p = Path(path)
    return self.add_sanskrit(read_text(p, kw.get('scheme')), title or p.stem.replace('_',' ').strip(),
                             source=str(p), store=store, prefix=prefix, **kw)

@patch
def add_sanskrit_dir(self:Database, dir, types:str=SANSKRIT_EXTS, store:str='store', prefix:str=None, **kw) -> list:
    'Ingest a directory of Sanskrit e-texts — a GRETIL mirror, say. Already-added sources are skipped.'
    exts = {f".{t.strip().lstrip('.')}".lower() for t in types.split(',')}
    return [self.add_sanskrit_file(p, store=store, prefix=prefix, **kw)
            for p in sorted(Path(dir).rglob('*')) if p.is_file() and p.suffix.lower() in exts]

# --- the query side ---------------------------------------------------------------------------
def sanskrit_query(q:str, wc:bool=True, min_len:int=2) -> str:
    """An FTS5 query that finds a Sanskrit phrase however the reader typed it.

    Every token is OR'd in three forms: as given, `fold`ed, and `loose`ned. `dharmakṣetre`,
    `dharmaksetre`, `धर्मक्षेत्रे` and `Krishna` therefore all reach the same chunks, because the
    chunk carries both its own script in `content` and its fold in `metadata`, and `get_store`
    indexes the two columns together. Each term is double-quoted, which is what makes a Devanagari
    token legal in an FTS5 expression at all."""
    toks = [t for t in re.split(r'[^\wÀ-ɏḀ-ỿऀ-ॿ]+', clean(q) or '') if t]
    terms = []
    for t in toks:
        for v in dict.fromkeys((t.lower(), fold(t), loose(t))):
            if len(v) >= min_len: terms.append('"' + v.replace('"', '""') + '"' + ('*' if wc else ''))
    return ' OR '.join(dict.fromkeys(terms))

def _md(row):
    'Parse a hit\'s `metadata` JSON onto the row, so `ref` and `meter` read like columns.'
    try: m = json.loads(row.get('metadata') or '{}')
    except Exception: m = {}
    return merge(row, dict(ref=m.get('ref',''), ref_end=m.get('ref_end',''), meter=m.get('meter'),
                           speaker=m.get('speaker'), nverses=m.get('nverses'), on=m.get('on'),
                           kind=m.get('kind')))

@patch
def verse_search(self:Database,
                 q:str,                 # query, in any transliteration
                 emb:bytes=None,        # query embedding; None runs the FTS leg alone
                 columns:list=None,
                 limit:int=10,
                 store:str='store',
                 prefix:str=None,
                 ref:str=None,          # restrict to citations starting with this prefix
                 meter:str=None,        # restrict to one metre
                 spans:bool=False,      # merge adjacent hits (off — see below)
                 where:str=None,
                 where_args:dict=None,
                 **kw                   # forwarded to `doc_search`
) -> list:
    """Hybrid search over a Sanskrit store, matching the query in every transliteration.

    `emb=None` is a first-class mode here, not a degradation. Looking up a half-remembered verse is
    a *lexical* act — the reader knows some of the words — and the fold makes that work across
    scripts without a model anywhere in the path.

    `spans` defaults **off**, the opposite of `doc_search`. Verse windows already overlap by
    `per - stride` verses, so merging two adjacent hits would concatenate the verses they share and
    hand the caller the same śloka twice."""
    cols = list(dict.fromkeys((columns or ['content']) + ['metadata','node_id','page','heading','doc_id','rowid']))
    wh, wa = [where] if where else [], dict(where_args or {})
    if ref:   wh.append('metadata like :sa_ref');   wa['sa_ref']   = f'% {ref}%'
    if meter: wh.append('metadata like :sa_meter'); wa['sa_meter'] = f'%"meter": "%{meter}%'
    wh = ' AND '.join(f'({w})' for w in wh) or None
    fq = sanskrit_query(q)
    if not fq: return []
    if emb is None:
        rows = self.t[store].fts_search(fq, cols, 'rank', limit, None, wh, wa, quote=False)
        for i, r in enumerate(rows): r['_rank'] = i
        return [_md(r) for r in rows]
    hits = self.doc_search(fq, emb, columns=cols, limit=limit, store=store, prefix=prefix,
                           spans=spans, where=wh, where_args=wa, quote=False, **kw)
    return [_md(h) for h in hits]

@patch
def by_ref(self:Database,
           ref:str,               # a citation, e.g. `ViP_1,1.1`
           store:str='store',
           prefix:str=None,
           limit:int=20) -> list:
    """The chunks whose verse window contains a citation. Exact lookup, no search involved.

    `sanskrit_chunks` writes the window's citations into `metadata` as a **space-padded** list
    (`" ViP_1,1.1 ViP_1,1.2 "`), which is the small trick that makes this a `LIKE` on a delimited
    field rather than a prefix match: without the padding, `ViP_1,1.1` also matches `ViP_1,1.10`."""
    rows = self.t[store](select='rowid as rowid, content, metadata, node_id, page, heading, doc_id',
                         where='metadata like :r', where_args=dict(r=f'% {ref} %'), limit=limit)
    return [_md(r) for r in rows]

In [ ]:
from litesearch.core import database
import tempfile
_d = Path(tempfile.mkdtemp())
# GRETIL serves its plain-text editions as .htm; `read_text` strips the markup rather than parsing it
(_d/'megha.htm').write_text("""<html><head><title>x</title></head><body>
<p>Kalidasa: Meghaduta<br>Based on the ed. of ... <br>
For a list of GRETIL encodings see http://gretil.sub.uni-goettingen.de/gretdiac.pdf</p>
<p>kaścit kāntāvirahaguruṇā svādhikārātpramattaḥ<br>
śāpenāstaṃgamitamahimā varṣabhogyeṇa bhartuḥ /<br>
yakṣaścakre janakatanayāsnānapuṇyodakeṣu<br>
snigdhacchāyātaruṣu vasatiṃ rāmagiryāśrameṣu // MD_1.1 //<br>
tasminn adrau katicid abalāviprayuktaḥ sa kāmī<br>
nītvā māsān kanakavalayabhraṃśariktaprakoṣṭhaḥ /<br>
āṣāḍhasya prathamadivase meghamāśliṣṭasānuṃ<br>
vaprakrīḍāpariṇatagajaprekṣaṇīyaṃ dadarśa // MD_1.2 //</p></body></html>""", encoding='utf-8')

db4 = database(':memory:')
db4.get_store('store', hash=True, doc_id=str, node_id=str, page=int, heading=str)
print(db4.add_sanskrit_file(_d/'megha.htm', 'Meghaduta', per=1, min_chunk=1))
assert [h['ref'] for h in db4.verse_search('meghamāśliṣṭasānuṃ')] == ['MD_1.2']
# the metre survived the round trip through the store, and is a filter
assert {h['meter'] for h in db4.verse_search('katicid OR kaścit')} == {'mandākrāntā'}
assert len(db4.verse_search('kāntāvirahaguruṇā', meter='mandākrāntā')) == 1
assert db4.verse_search('kāntāvirahaguruṇā', meter='anuṣṭubh') == []
assert 'Kalidasa: Meghaduta' not in ' '.join(r['content'] for r in db4.t.store(select='content'))

# a commentary block inherits the citation of the verse it glosses
COMM = """jyāyasī cet karmaṇas te matā buddhir janārdana |
tat kiṃ karmaṇi ghore māṃ niyojayasi keśava || BhG_3.1 ||
karma-yogasya jñāna-yogāpekṣayā śraiṣṭhyaṃ pratipādayiṣyan bhagavān arjunasya
praśnam avatārayati | jyāyasī cet iti | niścayena śreyaskarī ced ity arthaḥ |"""
db5 = database(':memory:')
db5.get_store('store', hash=True, doc_id=str, node_id=str, page=int, heading=str)
db5.add_sanskrit(COMM, 'BhG with gloss', header=False, per=1, min_chunk=1)
rows = db5.by_ref('BhG_3.1')
assert len(rows) == 2, rows                       # the verse and its gloss, both reachable by citation
assert {r['kind'] for r in rows} == {'verse', 'prose'}
# `on` records that the gloss is *about* the verse rather than *being* it
assert [r['on'] for r in rows if r['kind'] == 'prose'] == ['BhG_3.1']

## The graph: what connects verses

The design decision is one line: **every verse is itself an entity**, `kind='verse'`, named by its
citation. `get_graph`'s `edges` table is entity-to-entity and has no room for a chunk-to-chunk
link, so making the verse a node is what lets the four Sanskrit relations live in the schema that
already exists — and therefore lets `db.graph_search` walk them with no change.

| edge | means |
|---|---|
| `follows` | the next verse in reading order — a hit is rarely the whole answer |
| `parallel` | two verses are near-identical, usually across texts |
| `quotes` / `pratika` | a commentary cites a verse by its opening words |
| `chandas` | shares a metre |

**Verses are the unit, not chunks.** The store holds overlapping windows of several verses;
`chunk_units` splits them back apart, which is exact because each verse kept its closing daṇḍa and
its citation. Comparing windows instead would only find a parallel when two texts happened to
window the same verses together — which they do not — and it would make the PMI windows page-sized,
the mistake `graph.prose_windows` exists to avoid.

`parallel` is the one to care about. A Sanskrit verse is often not the property of the text you
found it in — it travels — and "this śloka is also Manusmṛti 2.1" is not a retrieval artefact but
the finding. `db.parallels(ref)` reads it back.

Note what is *dropped*: metres carried by more than `max_meter_df` of the corpus. Nine verses in ten
are anuṣṭubh, so a node for it is a clique over the whole store — the identical mistake
`graph._pmi_edges` guards against with its own `max_df`, and just as fatal.

In [ ]:
#| export
# Sanskrit function words, folded. These are the `ca`/`tu`/`hi`/`eva` that fill out a pāda for the
# metre and carry no content — the Sanskrit equivalent of `graph._STOP`, and needed for the same
# reason: without it every term-level co-occurrence edge in the corpus runs through `iti`.
SANSKRIT_STOP = frozenset('''ca va tu hi api iti eva evam atha atho tatha yatha yat tat tad sa sah
tam tan tena tasya tasmat tasmin tesam te tvam aham mam me nah vah asya ayam idam imam etat esa
esah kim kah ka ke na no ma vai khalu nu kila punah punar tada yada yatra tatra kutra sarva sarvam
sarve sarvani asti bhavati bhavet syat kuru krtva gatva uktva uvaca abravit aha bruyat ha sma
adi adyah ityadi ce ced yadi tarhi param paras parah tv anya anyat api ityevam tathaiti iva yad
asmat asmai atra ityuktva ityukte yasya yasmat yena tair tabhih tesu asmin ebhih ity'''.split())

_WORD = re.compile(r'[a-z]+')

def verse_terms(text,                 # verse or prose text, any transliteration
                scheme:str=None,
                min_len:int=4,        # shortest folded token kept
                stop=None,            # stopword set (defaults to `SANSKRIT_STOP`)
                bigrams:bool=True     # also emit adjacent content-word pairs
) -> L:
    """Content terms for one unit as `(surface, kind)`, with no model and no lexicon.

    Sanskrit gets no help from any of the usual extractors. spaCy has no model for it, sandhi means
    whitespace is not a word boundary, and compounding means a single "word" can be a whole clause,
    so noun-chunking is not available even in principle. What *is* available is the fold: a stable
    key per surface form. So the terms here are honest about what they are — **folded surface
    tokens and their adjacent pairs**, not lemmas — and `verse_graph`'s PMI pass is what separates
    the collocations that matter from the ones that are an accident of the metre.

    For real lemmas, use the `lemma_fn` seam on `add_sanskrit`, or ingest a DCS CoNLL-U file with
    `add_dcs`, where the lemmatisation has already been done and hand-validated."""
    st = SANSKRIT_STOP if stop is None else stop
    ws = [w for w in _WORD.findall(fold(text, scheme)) if len(w) >= min_len and w not in st]
    out = L((w, 'term') for w in dict.fromkeys(ws))
    if bigrams: out += L((f'{a} {b}', 'phrase') for a, b in dict.fromkeys(zip(ws, ws[1:])))
    return out

def pratika(text, words:int=3, min_len:int=10, scheme:str=None) -> str:
    """The *pratīka*: the opening words of a verse, folded.

    This is how the tradition itself refers to a verse — a commentator writes the first two or three
    words and expects you to know the rest — so it is both the natural handle for a verse node and
    the string to look for when deciding whether a piece of prose is quoting one."""
    ws = _WORD.findall(fold(text, scheme))
    p = ' '.join(ws[:words])
    return p if len(p) >= min_len else ' '.join(ws[:words+2])[:60]

def _shingles(txt, n:int=4):
    'Word n-grams of a folded string — the unit of comparison for a parallel passage.'
    ws = _WORD.findall(txt)
    if len(ws) < n: return {' '.join(ws)} if ws else set()
    return {' '.join(ws[i:i+n]) for i in range(len(ws)-n+1)}

def parallel_pairs(items,                  # [(key, folded_text)]
                   n:int=4,                # shingle width, in words
                   min_jaccard:float=0.5,  # overlap required to call it a parallel
                   max_df:int=40,          # ignore a shingle carried by more items than this
                   min_shared:int=2        # candidate pairs must share at least this many shingles
) -> L:
    """Near-identical passages across a corpus, as `(a, b, jaccard)`.

    This is the offline, lexical cousin of what Dharmamitra's MITRA pipeline does with cross-lingual
    embeddings, and within one language it is the cheaper tool for the job: Sanskrit parallels are
    usually **verbatim or nearly so** — a śloka lifted from the Manusmṛti into the Mahābhārata, a
    verse a commentator quotes from three works back — and the fold already removes the orthographic
    and sandhi noise that would otherwise hide the identity.

    `max_df` is the same lesson `graph._pmi_edges` learned about hub terms: a shingle shared by
    hundreds of verses is a metrical formula (`iti me matih`), and left in the inverted index it
    makes the candidate generation quadratic and the results meaningless."""
    sh = {k: _shingles(t, n) for k, t in items}
    inv = {}
    for k, s in sh.items():
        for g in s: inv.setdefault(g, []).append(k)
    cand = {}
    for g, ks in inv.items():
        if len(ks) > max_df or len(ks) < 2: continue
        for i in range(len(ks)):
            for j in range(i+1, len(ks)):
                p = (ks[i], ks[j]) if ks[i] < ks[j] else (ks[j], ks[i])
                cand[p] = cand.get(p, 0) + 1
    out = L()
    for (a, b), c in cand.items():
        if c < min_shared: continue
        A, B = sh[a], sh[b]
        if not (A and B): continue
        jac = len(A & B) / len(A | B)
        if jac >= min_jaccard: out.append((a, b, round(jac, 4)))
    return out.sorted(key=lambda t: -t[2])

def chunk_units(content, refs=()) -> L:
    """Recover `(ref, verse_text)` per verse from a stored chunk.

    The graph works one verse at a time, and the store holds windows of several — but nothing is
    lost, because `split_verses` keeps each verse's closing daṇḍa in its text and
    `sanskrit_chunks` writes the citations in reading order. Splitting the window back apart is
    therefore exact, and it is what makes `parallel` mean anything: comparing *windows* only finds a
    parallel when two texts happen to have windowed the same verses together, which they do not."""
    parts, out = _DANDA2.split(content or ''), L()
    for i in range(0, len(parts), 2):
        t = parts[i].strip()
        if not t: continue
        out.append(f'{t} {parts[i+1]}'.strip() if i+1 < len(parts) else t)
    rs = [r for r in refs if r]
    if len(rs) == len(out) and out: return L(zip(rs, out))
    return L([(rs[0] if rs else '', (content or '').strip())])    # prose, or a window we cannot align

def verse_graph(db,                       # Database with a Sanskrit store already ingested
                store:str='store',
                prefix:str=None,
                terms:bool=True,          # term/phrase entities plus PMI co-occurrence edges
                sequence:bool=True,       # `follows` edges along the reading order of each section
                meter:bool=True,          # a `chandas` node per metre, rare metres only
                parallel:bool=True,       # `parallel` edges between near-identical verses
                quotes:bool=True,         # `quotes` edges from prose that cites a verse's pratika
                min_jaccard:float=0.5,    # overlap required for a parallel
                max_meter_df:float=0.3,   # skip a metre carried by more than this share of the corpus
                min_n:int=2,              # PMI: minimum co-occurrence count
                min_npmi:float=0.15,      # PMI: minimum normalized PMI
                max_df:float=0.4,         # PMI: drop terms present in more than this share of verses
                max_degree:int=48,
                emb_fn=None,              # embedder for entity names (needed by `resolve_entities`)
                where:str=None
) -> dict:
    """Build the entity graph for a Sanskrit store, in the tables `get_graph` already defines.

    The design decision worth stating: **every verse is itself an entity**, `kind='verse'`, whose
    name is its citation. That is what lets the four Sanskrit-specific relations live in the
    existing `edges` table, which is entity-to-entity and has no room for a chunk-to-chunk link:

    | edge | means | why it earns its place |
    |---|---|---|
    | `follows` | the next verse in reading order | a hit is rarely the whole answer; the argument continues |
    | `parallel` | two verses are near-identical | the same śloka is in four texts, and that is the finding |
    | `quotes` | a commentary cites a verse by its pratīka | links a gloss to its root across works |
    | `chandas` | shares a metre | reaches verses no shared vocabulary would |

    Because those are ordinary edges, `db.graph_search` walks them with no change: a hit spreads
    PageRank mass to its parallels, its neighbours and its commentaries, which come back as chunks
    through the same `mentions` join everything else uses.

    Verses, not chunks, are the unit throughout — see `chunk_units`. It also makes the PMI windows
    tight, which is the same lesson `graph.prose_windows` learned about sentences: a page-sized
    window turns every pair of terms into a clique and no amount of pruning recovers from it.

    Metres above `max_meter_df` are dropped rather than linked. Roughly nine verses in ten are
    anuṣṭubh, so a `chandas` node for it is a clique over the corpus — the identical mistake
    `_pmi_edges` guards against with its own `max_df`, and just as fatal here."""
    g = db.get_graph(store, prefix, ann=bool(emb_fn))
    rows = L(db.t[store](select='id, content, metadata, node_id, doc_id, page', where=where))
    if not rows: return dict(entities=0, mentions=0, edges=0, verses=0)
    ents, mens, edges, wins = {}, {}, {}, []
    def ent(name, kind):
        n = _WS.sub(' ', (name or '').strip())[:60]
        if not n: return None
        i = _slug(n)
        e = ents.setdefault(i, dict(content=n, kind=kind, freq=0, canon=i))
        e['freq'] += 1
        return i
    def men(cid, eid, surface):
        if not (cid and eid): return
        m = mens.setdefault((cid, eid), dict(chunk_id=cid, entity_id=eid, surface=(surface or '')[:60], n=0))
        m['n'] += 1
    def edge(s, d, rel, w=1.0):
        if not (s and d) or s == d: return
        e = edges.setdefault((s, d, rel), dict(src=s, dst=d, rel=rel, weight=0.0, n=0))
        e['weight'] = max(e['weight'], w); e['n'] += 1

    # one record per verse, deduplicated by citation: overlapping windows repeat verses, and a verse
    # that appears in two windows is one verse
    info, mcount, seen = L(), Counter(), {}
    for r in rows:
        try: md = json.loads(r['metadata'] or '{}')
        except Exception: md = {}
        prose = md.get('kind') == 'prose'
        for k, (ref, txt) in enumerate(chunk_units(r['content'], (md.get('refs') or '').split())):
            key = ref or f'{r["node_id"]}@{r["page"]}#{k}'
            vid = ent(key, 'verse')
            men(r['id'], vid, ref)
            if vid in seen: seen[vid].chunks.append(r['id']); continue
            m = detect_meter(txt) if (meter and not prose) else None
            if m and m.name: mcount[m.name] += 1
            it = AttrDict(key=key, vid=vid, text=txt, fold=fold(txt), prose=prose, meter=(m.name if m else None),
                          node=r['node_id'], doc=r['doc_id'], page=r['page'], order=(r['page'] or 0, k),
                          chunks=[r['id']])
            seen[vid] = it; info.append(it)
    nv = max(1, len(info))
    if sequence:
        by_node = {}
        for it in info: by_node.setdefault(it.node, []).append(it)
        for its in by_node.values():
            its.sort(key=lambda x: x.order)
            for a, b in zip(its, its[1:]): edge(a.vid, b.vid, 'follows')
    if meter:
        keep = {m for m, c in mcount.items() if c/nv <= max_meter_df}
        for it in info:
            if it.meter in keep:
                i = ent(it.meter, 'chandas')
                for c in it.chunks: men(c, i, it.meter)
                edge(it.vid, i, 'chandas')
    if terms:
        for it in info:
            w = set()
            for surf, kind in verse_terms(it.fold):
                if (i := ent(surf, kind)):
                    for c in it.chunks: men(c, i, surf)
                    w.add(i)
            if len(w) > 1: wins.append(w)
        for e in _pmi_edges(wins, min_n, min_npmi, max_df, max_degree):
            edges[(e['src'], e['dst'], e['rel'])] = e
    npar = nq = 0
    if parallel:
        for a, b, j in parallel_pairs([(it.vid, it.fold) for it in info if not it.prose],
                                      min_jaccard=min_jaccard):
            edge(a, b, 'parallel', j); npar += 1
    if quotes:
        prat = {p: it.vid for it in info if not it.prose and (p := pratika(it.fold))}
        for it in info:
            if not it.prose: continue
            for p, vid in prat.items():
                if p and p in it.fold and vid != it.vid:
                    i = ent(p, 'pratika')
                    for c in it.chunks: men(c, i, p)
                    edge(it.vid, vid, 'quotes'); edge(i, vid, 'pratika'); nq += 1
    rws = list(ents.values())
    if rws:
        if emb_fn: process_content(g.entities, rws, embed=True, emb_fn=emb_fn)
        else:      g.entities.insert_all(rws, upsert=True, hash_id='id', hash_id_columns=['content'])
    if mens:  g.mentions.insert_all(list(mens.values()), upsert=True, pk=('chunk_id','entity_id'))
    if edges: g.edges.insert_all(list(edges.values()), upsert=True, pk=('src','dst','rel'))
    if emb_fn and rws: g.entities.rebuild_index()
    return dict(entities=len(rws), mentions=len(mens), edges=len(edges), verses=len(info),
                parallels=npar, quotes=nq, windows=len(wins),
                meters={m: c for m, c in mcount.most_common()})

@patch
def parallels(self:Database,
              ref:str,                  # a citation, e.g. `Mn_2.1`
              store:str='store',
              prefix:str=None,
              rels:str='parallel,quotes',
              limit:int=20) -> list:
    """The verses linked to a citation — the same śloka elsewhere, or the commentaries that quote it.

    The single most useful thing the graph adds for this corpus. A Sanskrit verse is very often not
    the property of the text you found it in: it travels, and the fact that Manusmṛti 2.1 also sits
    in the Mahābhārata is not a retrieval artefact but the thing a reader wants to be told."""
    g = self.get_graph(store, prefix, ann=False)
    rel = [r.strip() for r in rels.split(',') if r.strip()]
    src = first(g.entities(where='content=:c', where_args=dict(c=ref)))
    if not src: return []
    rows = g.edges(where=f"({_in('src', [src['id']])} OR {_in('dst', [src['id']])}) AND {_in('rel', rel)}")
    ids = {(r['dst'] if r['src'] == src['id'] else r['src']): r for r in rows}
    if not ids: return []
    ents = {e['id']: e for e in g.entities(where=_in('id', list(ids)))}
    out = [dict(ref=ents[i]['content'], rel=r['rel'], weight=r['weight'], kind=ents[i]['kind'])
           for i, r in ids.items() if i in ents]
    return sorted(out, key=lambda d: -(d['weight'] or 0))[:limit]

In [ ]:
# a window of verses is split back into its verses exactly, because each one kept its daṇḍa
_c = 'aaa bbb /\nccc ddd //\neee fff /\nggg hhh //'
assert chunk_units(_c, ['X_1', 'X_2']) == [('X_1', 'aaa bbb /\nccc ddd //'), ('X_2', 'eee fff /\nggg hhh //')]
# a count mismatch (prose, or a window we cannot align) degrades to one unit rather than misaligning
assert len(chunk_units(_c, ['X_1'])) == 1

# a verbatim parallel is found even when the two texts window their verses differently
assert [j for _, _, j in parallel_pairs([('a', 'kim api vacanam idam uktam iha tena'),
                                         ('b', 'kim api vacanam idam uktam iha tena'),
                                         ('c', 'anyad eva khalu bhavati vacanam etat')])] == [1.0]
assert pratika('kim api vacanam idam uktam') == 'kim api vacanam'
assert 'ca' in SANSKRIT_STOP and 'iti' in SANSKRIT_STOP
assert [s for s, k in verse_terms('dharmam eva ca vaksyami tatra')] == ['dharmam', 'vaksyami', 'dharmam vaksyami']

## Lemmas: the Digital Corpus of Sanskrit

Everything above is deterministic string work. The one thing that genuinely needs linguistics is
**word segmentation**, and it is also the thing that matters most: sandhi and compounding mean the
surface form of a Sanskrit word is frequently not the form anyone will type. Stemming is a
refinement in English; segmentation is the difference between finding the verse and not.

Two seams, no dependency. `lemma_fn=` on `add_sanskrit` takes any `list[str] -> list[str]` — the
ByT5-Sanskrit segmenter, or your own. And `add_dcs` ingests a
[DCS](https://github.com/OliverHellwig/sanskrit/tree/master/dcs/data/conllu) CoNLL-U file directly,
indexing its hand-validated lemmas beside the surface text.

In [ ]:
#| export
def load_conllu(path_or_text) -> L:
    """Sentences from a CoNLL-U file as `AttrDict(sent_id, text, forms, lemmas, upos, meta)`.

    The Digital Corpus of Sanskrit publishes its whole annotation this way, and it is the one
    Sanskrit resource that changes retrieval quality outright rather than incrementally: 600,000+
    lines with **hand-validated** segmentation and lemmas. Sandhi and compounding mean the surface
    form of a Sanskrit word is very often not the form a reader will type — `tac ca` for `tat ca`,
    `rāmo 'gacchat` for `rāmaḥ agacchat` — so a lemma index is not a refinement of surface FTS here
    the way stemming is in English. It is the difference between finding the verse and not."""
    txt = path_or_text
    if isinstance(txt, (str, Path)) and '\n' not in str(txt) and Path(txt).exists():
        txt = Path(txt).read_text(encoding='utf-8', errors='replace')
    out, meta, rows = L(), {}, []
    def flush():
        if not rows and not meta: return
        forms  = [r[1] for r in rows]
        lemmas = [r[2] for r in rows if len(r) > 2 and r[2] not in ('_', '')]
        out.append(AttrDict(sent_id=meta.get('sent_id') or meta.get('sent_counter') or str(len(out)+1),
                            text=meta.get('text') or meta.get('text_line') or ' '.join(forms),
                            forms=forms, lemmas=lemmas,
                            upos=[r[3] for r in rows if len(r) > 3], meta=dict(meta)))
        meta.clear(); rows.clear()
    for ln in (txt or '').splitlines():
        s = ln.rstrip('\n')
        if not s.strip(): flush(); continue
        if s.startswith('#'):
            k, _, v = s.lstrip('#').partition('=')
            if v: meta[k.strip()] = v.strip()
            continue
        f = s.split('\t')
        if len(f) >= 2 and '-' not in f[0]: rows.append(f)
    flush()
    return out

_NUMS = re.compile(r'\d+')

def dcs_verses(sents,                 # sentences from `load_conllu`
               siglum:str='',         # citation prefix, e.g. `Mn`
               ref_keys=('ref','verse','sent_id','sent_counter'),
               chapter_keys=('chapter','book','adhyaya'),
               meter:bool=True
) -> L:
    """CoNLL-U sentences turned into verse records, with their lemmas attached.

    The address is rebuilt from whatever the file's comment lines actually carry — DCS names the
    chapter, and the sentence counter within it — so `verse_tree` sees the same two-level reference
    it would get from a GRETIL `Mn_1.1` and builds the same tree."""
    out, seen = L(), Counter()
    for s in sents:
        ch = first((_NUMS.findall(s.meta[k]) for k in chapter_keys if s.meta.get(k)), None)
        ch = int(ch[0]) if ch else None
        vn = first((_NUMS.findall(str(s.meta.get(k) or '')) for k in ref_keys if s.meta.get(k)), None)
        if ch is not None: seen[ch] += 1
        vn = int(vn[-1]) if vn else seen[ch or 0]
        ref = format_ref(siglum, (ch, vn) if ch is not None else (vn,), (',',) if ch is not None else ())
        u = _mk('verse', s.text, 0, ref)
        u.padas = [x.strip() for x in _DANDA1.split(s.text) if x.strip()]
        u.lemmas = ' '.join(s.lemmas)
        if meter:
            w = [g for _, g in syllables(s.text)]
            u.syllables = len(w)
            m = detect_meter(s.text, weights=w)
            u.meter = m.name if m else None
            if not is_verse(s.text, weights=w): u.kind = 'prose'
        out.append(u)
    for i, u in enumerate(out): u.pos = i
    return out

@patch
def add_dcs(self:Database,
            path,                  # a `.conllu` file (or its text)
            title:str=None,
            siglum:str='',         # citation prefix for the rebuilt references
            store:str='store',
            prefix:str=None,
            **kw                   # forwarded to `add_sanskrit`
) -> dict:
    'Ingest a Digital Corpus of Sanskrit CoNLL-U file, indexing its validated lemmas beside the text.'
    vs = dcs_verses(load_conllu(path), siglum)
    ttl = title or (Path(path).stem.replace('_',' ') if isinstance(path, (str, Path)) else 'DCS text')
    src = str(path) if isinstance(path, (str, Path)) and len(str(path)) < 400 else ttl
    return self.add_sanskrit('\n'.join(v.text for v in vs), ttl, source=src, kind='dcs',
                             store=store, prefix=prefix, verses=vs, header=False, **kw)

In [ ]:
# a DCS CoNLL-U fragment: the lemmas are indexed beside the surface text
CONLLU = '''# chapter = 1
# sent_counter = 1
# text_line = tac ca rāmo 'gacchat
1\ttac\ttad\tPRON\t_\t_\t0\troot\t_\t_
2\tca\tca\tCCONJ\t_\t_\t1\tcc\t_\t_
3\trāmo\trāma\tNOUN\t_\t_\t1\tnsubj\t_\t_
4\t'gacchat\tgam\tVERB\t_\t_\t1\troot\t_\t_

# chapter = 1
# sent_counter = 2
# text_line = sītā vanam agamat
1\tsītā\tsītā\tNOUN\t_\t_\t0\troot\t_\t_
2\tvanam\tvana\tNOUN\t_\t_\t3\tobj\t_\t_
3\tagamat\tgam\tVERB\t_\t_\t0\troot\t_\t_
'''
sents = load_conllu(CONLLU)
assert len(sents) == 2 and sents[0].lemmas == ['tad', 'ca', 'rāma', 'gam']
vs = dcs_verses(sents, siglum='Ram')
assert [v.ref for v in vs] == ['Ram_1,1', 'Ram_1,2'] and vs[0].lemmas == 'tad ca rāma gam'

from litesearch.core import database
db2 = database(':memory:')
db2.get_store('store', hash=True, doc_id=str, node_id=str, page=int, heading=str)
print(db2.add_dcs(CONLLU, 'Ramayana sample', siglum='Ram', per=1))
# `gam` is the lemma of both `'gacchat` and `agamat`, and neither surface form contains it
assert not any(w == 'gam' for w in fold("tac ca rāmo 'gacchat").split())
assert [h['ref'] for h in db2.verse_search('gam', limit=5)] == ['Ram_1,1']
# these two sentences were short enough to merge, and the merge kept *both* citations
assert db2.by_ref('Ram_1,1') and db2.by_ref('Ram_1,2')
# with a lower floor they stay apart, and each is found by its own lemma
db3 = database(':memory:')
db3.get_store('store', hash=True, doc_id=str, node_id=str, page=int, heading=str)
db3.add_dcs(CONLLU, 'Ramayana sample', siglum='Ram', per=1, min_chunk=1)
assert sorted(h['ref'] for h in db3.verse_search('gam', limit=5)) == ['Ram_1,1', 'Ram_1,2']
assert [h['ref'] for h in db3.verse_search('vana', limit=5)] == ['Ram_1,2']   # lemma, not `vanam`

## End to end

Two texts, one of which quotes the other verbatim — which is the ordinary case in this
corpus, not a contrivance. A deterministic hash embedder stands in for a real model so the
notebook runs offline.

In [ ]:
import numpy as np, hashlib
from litesearch.core import database
from litesearch.utils import doc_encoder

def _emb(txts, ndim=64, **kw):
    'A deterministic hash embedder -- no download, no model, so this notebook runs offline.'
    out = np.zeros((len(txts), ndim), dtype=np.float32)
    for i, t in enumerate(txts):
        for w in fold(t).split():
            out[i, int(hashlib.md5(w.encode()).hexdigest(), 16) % ndim] += 1.0
    out /= np.clip(np.linalg.norm(out, axis=1, keepdims=True), 1e-9, None)
    return out.astype(np.float16)

MANU = '''manum ekāgram āsīnam abhigamya maharṣayaḥ /
pratipūjya yathānyāyam idaṃ vacanam abruvan // Mn_1.1 //
bhagavan sarvavarṇānāṃ yathāvad anupūrvaśaḥ /
antaraprabhavānāṃ ca dharmān no vaktum arhasi // Mn_1.2 //
[[iti manusmṛtau prathamo 'dhyāyaḥ ]]
vidvadbhiḥ sevitaḥ sadbhir nityam adveṣarāgibhiḥ /
hṛdayenābhyanujñāto yo dharmas taṃ nibodhata // Mn_2.1 //
akāmasya kriyā kā cid dṛśyate neha karhicit /
yad yad hi kurute kiṃ cit tat tat kāmasya ceṣṭitam // Mn_2.4 //
[[iti manusmṛtau dvitīyo 'dhyāyaḥ ]]'''
# the same two ślokas travel into the epic, verbatim -- which is the point of `parallel`
MBH = '''vidvadbhiḥ sevitaḥ sadbhir nityam adveṣarāgibhiḥ /
hṛdayenābhyanujñāto yo dharmas taṃ nibodhata // MBh_1,1.1 //
akāmasya kriyā kā cid dṛśyate neha karhicit /
yad yad hi kurute kiṃ cit tat tat kāmasya ceṣṭitam // MBh_1,1.2 //
dharmakṣetre kurukṣetre samavetā yuyutsavaḥ /
māmakāḥ pāṇḍavāś caiva kim akurvata sañjaya // MBh_1,1.3 //'''

db = database(':memory:')
db.get_store('store', hash=True, ann=True, doc_id=str, node_id=str, page=int, heading=str)
print(db.add_sanskrit(MANU, 'Manusmrti',  header=False, emb_fn=_emb, per=2, stride=1))
print(db.add_sanskrit(MBH,  'Mahabharata', header=False, emb_fn=_emb, per=2, stride=1))
db.toc('Manusmrti')

In [ ]:
# the tree, read back with the machinery that was already there
for d in db.toc('Manusmrti'):
    print(d['title'], '->', [(c['title'], c['pages']) for c in d['tree'].get('children', [])])
print(db.breadcrumb(db.toc('Manusmrti')[0]['tree']['children'][1]['id']))

In [ ]:
# search finds the verse in any transliteration -- and with no embedder at all
for q in ('adveṣarāgibhiḥ', 'advesaragibhih', 'अद्वेषरागिभिः', 'nityam advesa'):
    hits = db.verse_search(q, limit=3)
    print(f'{q:22s} -> {[h["ref"] for h in hits]}')
    assert any(h['ref'].startswith(('Mn_2', 'MBh_1')) for h in hits), q

# the hybrid leg, and the exact lookup
h = db.verse_search('kāmasya ceṣṭitam', _emb(['kāmasya ceṣṭitam'])[0].tobytes(), limit=3)
print([(x['ref'], x['breadcrumb']) for x in h])
assert db.by_ref('Mn_2.4') and all('Mn_2.4' in json.loads(r['metadata'])['refs'] for r in db.by_ref('Mn_2.4'))
assert not db.by_ref('Mn_2.40')          # the padding is what stops 2.4 matching 2.40

In [ ]:
# the graph: every verse is a node, so `parallel` fits the edge table that already existed
print(verse_graph(db, emb_fn=_emb, min_jaccard=0.6))
par = db.parallels('Mn_2.1')
print('Mn_2.1 also appears as:', par)
assert any(p['ref'].startswith('MBh_') and p['rel'] == 'parallel' for p in par)
assert any(p['ref'].startswith('Mn_') for p in db.parallels('MBh_1,1.1'))   # and back the other way

from litesearch.graph import graph_stats
print(graph_stats(db))
# `follows` links the reading order, so a hit can be continued
assert [e['rel'] for e in db.t.edges(where="rel='follows'")]

## What this does not do

- **No sandhi splitting.** `verse_terms` is honest about extracting *folded surface tokens*, not
  lemmas. That is a real ceiling on the term graph, and `lemma_fn` / `add_dcs` are how you lift it.
- **No cross-lingual retrieval.** Finding the Tibetan or Chinese parallel of a Sanskrit passage
  needs an aligned multilingual encoder; `parallel_pairs` is lexical and within-language by
  construction. Point `emb_fn` at a MITRA-style model if you need that leg.
- **Not measured against a Sanskrit relevance set.** The structural claims here are verifiable and
  are tested above — verse boundaries, references, metres, parallels. The *retrieval* claims are
  reasoned from `07_doc_eval`'s finding about granularity, not measured on this corpus, and
  `per`/`stride`/`emb_text` should be swept with that notebook before being trusted.
- **`emb_text` is a knob, not a recommendation.** See `sanskrit_embed`.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()